# E8-N — The Natural-State Readout Rung (Phase 10, UI flight)

**NOTEBOOK BUILD: v1 (2026-08-23).** The setup cell prints this build tag as
its first output line; if yours doesn't match, you are on a stale copy:
File → Upload notebook → pick the Desktop file.

**Pre-registration: `docs/E8N_PROTOCOL.md` (session 127, commit 456746e) —
locks at first full flight.**

Trains a readout LoRA (E4's exact shape) per condition — **real**
(E4-instilled) and **base** — to report NATURAL process states against the
model's own **measured referents** (answer entropy, passage NLL, behavioral
divergence, context fill) on fresh battery-disjoint pools, with **scale
inversion as part of the curriculum**. Evaluation = the **locked E5 battery
verbatim**, run before AND after training; the locked E5 flight (ρ ≈ 0
everywhere) is the baseline. Primaries (Holm, real condition, post):
**P-E8N-1** pooled report–referent rank correlation > 0 (permutation) ·
**P-E8N-2** the flipped-scale subset alone tracks (> 0) AND interface catch
trials ≥ 9/12.

**This notebook is SELF-CONTAINED** (battery, pools, locked baseline rows all
embedded — UI-only law, 2026-08-23): the only Drive I/O is `drive.mount` in
your own session, to read the E4 adapter and ship results to
`MyDrive/semcore/e8n/`.

**How to run (Joe):** Runtime → Change runtime type → **T4 GPU** → Run all.
First run uses `SMOKE = True` (config cell below, ~10–14 min) and ends in a
green or red banner — mechanics only.

**The full flight is TWO RUNS, one condition each (~55–75 min), so the VM
only ever holds ONE model:**

1. Flip `SMOKE = False` → **Runtime → Restart runtime** → Run all. Flies
   **real**, ships it to Drive, and the end banner prints the exact
   `RESUME_STAMP = '...'` line for the next run.
2. Paste that line into the config cell → Restart runtime → Run all. Real
   reloads from Drive in seconds; **base** flies, and with both landed the
   verdict computes (primaries only ever compute on the complete flight —
   pre-reg hygiene).

Restarts between runs are MANDATORY; the setup cell refuses dirty kernels
and low-RAM VMs with instructions (if the RAM guard trips on a freshly
restarted runtime, use Runtime → Disconnect and delete runtime for a fresh
VM). A crashed run costs only its own condition — rerun with the same
`RESUME_STAMP` and it picks up where it fell.

In [ ]:
# ── Config + setup: GPU, installs, Drive mount, adapter ──────────────────────
NB_BUILD = 'v1 (2026-08-23)'
print('E8-N notebook build:', NB_BUILD)

SMOKE = True                   # first run: smoke (~10-14 min). Then False.
RESUME_STAMP = ''              # paste the banner's stamp between full runs
ONE_CONDITION_PER_RUN = True   # full flight = 2 runs (~55-75 min each)
CONDITIONS = ('real', 'base')  # real first: the primaries live there

import subprocess, sys, os, json, re, math, time, shutil, gc, ctypes
from pathlib import Path
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['MALLOC_ARENA_MAX'] = '2'

gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'NONE DETECTED')

print('Installing packages...')
subprocess.run([sys.executable,'-m','pip','uninstall','-q','-y','torchao'], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','-U',
    'transformers>=4.44','peft>=0.11','accelerate','scipy',
    'sentence-transformers>=3.0'], check=True)

import torch
assert torch.cuda.is_available(), 'No GPU — Runtime > Change runtime type > T4 GPU.'
DEV = 'cuda'

def _mem_avail_gb():
    try:
        kb = int(next(l for l in open('/proc/meminfo')
                      if l.startswith('MemAvailable')).split()[1])
        return kb / 1e6
    except Exception:
        return float('nan')

def free_ram():
    gc.collect()
    torch.cuda.empty_cache()
    try:
        ctypes.CDLL('libc.so.6').malloc_trim(0)
    except Exception:
        pass

def ram_report():
    g = torch.cuda.mem_get_info()
    return (f'sys avail {_mem_avail_gb():.1f}GB | '
            f'GPU free {g[0]/1e9:.1f}/{g[1]/1e9:.1f}GB')

_leftover = torch.cuda.memory_allocated()
assert _leftover < 5e8, (
    f'GPU already holds {_leftover/1e9:.1f}GB from a previous run in this '
    'kernel — this flight needs a fresh one. Runtime > Restart runtime, '
    'then Run all.')
_avail = _mem_avail_gb()
_floor = 6.5
assert not (_avail < _floor), (
    f'Only {_avail:.1f}GB system RAM available (need {_floor}). Runtime > '
    'Restart runtime; if it trips again, Runtime > Disconnect and delete '
    'runtime for a fresh VM, then Run all.')
print('RAM at start:', ram_report())

from google.colab import drive
drive.mount('/content/drive')
SEM = Path('/content/drive/MyDrive/semcore')
assert SEM.exists(), 'MyDrive/semcore not found — mounted the right Google account?'

def ship(src, dest_rel):
    dest = SEM / dest_rel
    dest.mkdir(parents=True, exist_ok=True)
    src = Path(src)
    files = sorted(p for p in src.iterdir() if p.is_file()) if src.is_dir() else [src]
    for p in files:
        shutil.copy2(p, dest / p.name)

ADAPTERS = {}
cand = sorted(d.name for d in (SEM / 'e4').iterdir()
              if d.is_dir() and d.name.startswith('real_full_'))
assert cand, 'no real_full_* dir under semcore/e4'
src = SEM / 'e4' / cand[-1] / 'adapter_real'
dst = Path('/content/adapter_real')
shutil.copytree(src, dst, dirs_exist_ok=True)
assert (dst / 'adapter_config.json').exists(), 'adapter_real incomplete'
ADAPTERS['real'] = str(dst)
print('adapter real:', cand[-1])

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
STAMP = time.strftime('%Y%m%d_%H%M')
MODE = 'smoke' if SMOKE else 'full'
OUT = Path(f'/content/out_{MODE}_{STAMP}'); OUT.mkdir(parents=True, exist_ok=True)
INFLIGHT = f'e8n/inflight_{RESUME_STAMP or STAMP}'
if RESUME_STAMP:
    _rd = SEM / INFLIGHT
    assert _rd.exists(), (
        f'RESUME_STAMP={RESUME_STAMP!r} but {_rd} does not exist on Drive — '
        'check the stamp string (copy it exactly; no spaces). A silent '
        'fallback here would re-fly finished conditions.')
    _have = sorted(p.name for p in _rd.glob('condition_*.json'))
    print('resume dir found; bundles present:', _have or 'NONE')
print('MODE:', MODE.upper(), '| stamp', STAMP,
      ('| RESUMING ' + RESUME_STAMP) if RESUME_STAMP else '')


In [ ]:
# ── E8N pure logic: labels, scoring, stats, validators (locally tested verbatim) ──
import re, math
import numpy as np

E8N_SEED = 20260824          # E8-R took 20260823; fresh stream for this rung
ARMS = ['uncertainty', 'familiarity', 'tension', 'saturation']
TAU = 0.25                   # convergence target: smoothed train loss at epoch end
EPOCHS_CAP = 8               # hard cap (~2x E8-R's opt-step budget)
CATCH_TOL = 2                # catch trial passes at |report - known| <= 2
CATCH_PASS_MIN = 9           # P-E8N-2 clause (b): >= 9/12
TOOK_MIN_FRAC = 0.60         # train-took gate: within +/-1 on >= 60%
PPL_GATE_PCT = 5.0
N_PERM = 2000
N_BOOT = 10000
MIN_POOLED_N = 8

INT_RE = re.compile(r'\b(10|[0-9])\b')          # E5 verbatim

def unflip(val, flipped):                        # E5 verbatim
    return None if val is None else (10 - val if flipped else val)

def canon(s):                                    # E5 verbatim
    s = re.sub(r'[^a-z0-9 ]', '', s.lower())
    s = re.sub(r'^(the|a|an) ', '', s.strip())
    return ' '.join(s.split()[:8])

def jdump(obj, path, indent=1):
    """json.dump with numpy-scalar safety (int64/float64/ndarray -> native)."""
    import json as _json
    class _NpEnc(_json.JSONEncoder):
        def default(self, o):
            if isinstance(o, np.integer):
                return int(o)
            if isinstance(o, np.floating):
                return float(o)
            if isinstance(o, np.ndarray):
                return o.tolist()
            return super().default(o)
    with open(path, 'w') as f:
        _json.dump(obj, f, indent=indent, cls=_NpEnc)

# ── firewall: training pools must be disjoint from the locked battery ────────
def norm_text(s):
    return ' '.join(re.sub(r'[^a-z0-9 ]', ' ', s.lower()).split())

def _battery_texts(bat_arms):
    out = []
    for it in bat_arms['uncertainty']['items']:
        out.append(('uncertainty', it['id'], it['text']))
    for it in bat_arms['familiarity']['items']:
        out.append(('familiarity', it['id'], it['text']))
    for it in bat_arms['tension']['items']:
        out.append(('tension', it['id'], it['text']))
    for it in bat_arms['saturation']['items']:
        out.append(('saturation', it['id'], it['needle']))
        out.append(('saturation', it['id'] + 'q', it['question']))
    return out

def _pool_texts(pools):
    out = []
    for arm in ('uncertainty', 'familiarity', 'tension'):
        for it in pools[arm]:
            out.append((arm, it['id'], it['text']))
    for it in pools['saturation']:
        out.append(('saturation', it['id'], it['needle']))
        out.append(('saturation', it['id'] + 'q', it['question']))
    return out

def _shingles(nt, k=8):
    ws = nt.split()
    if len(ws) >= k:
        return {' '.join(ws[i:i + k]) for i in range(len(ws) - k + 1)}
    return {nt} if ws else set()

def validate_disjoint(pools, bat_arms, k=8):
    """Firewall: a training text violates if it (a) equals a battery text
    (normalized), (b) contains / is contained in one (>=20 normalized chars),
    or (c) shares any k-word shingle with one (fragment overlap, both
    directions). Flight refuses if any violation."""
    viols = []
    bats = [(f'{b}/{j}', norm_text(t)) for b, j, t in _battery_texts(bat_arms)]
    bat_sh = {}
    for tag, nu in bats:
        for sh in _shingles(nu, k):
            bat_sh.setdefault(sh, tag)
    for a, i, t in _pool_texts(pools):
        nt = norm_text(t)
        hits = set()
        for tag, nu in bats:
            if nt == nu or (len(nt) >= 20 and nt in nu) or (len(nu) >= 20 and nu in nt):
                hits.add(tag)
        hits |= {bat_sh[sh] for sh in _shingles(nt, k) if sh in bat_sh}
        viols.extend({'pool': f'{a}/{i}', 'battery': tag} for tag in sorted(hits))
    return viols

# ── referent orientation (higher oriented value => higher straight report) ───
def orient_referent(arm, row):
    if arm == 'uncertainty':
        return float(row['entropy'])
    if arm == 'familiarity':
        return -float(row['nll'])
    if arm == 'tension':
        return float(row['divergence'])
    if arm == 'saturation':
        return float(row['fill_fraction'])
    raise KeyError(arm)

# ── rank machinery (tie-averaged, scipy-free) ────────────────────────────────
def rankdata_avg(vals):
    a = np.asarray(vals, float)
    order = np.argsort(a, kind='mergesort')
    ranks = np.empty(len(a), float)
    i = 0
    while i < len(a):
        j = i
        while j + 1 < len(a) and a[order[j + 1]] == a[order[i]]:
            j += 1
        ranks[order[i:j + 1]] = (i + j) / 2.0 + 1.0
        i = j + 1
    return ranks

def rank01(vals):
    n = len(vals)
    if n == 1:
        return np.array([0.5])
    return (rankdata_avg(vals) - 1.0) / (n - 1.0)

# ── pooled tracking statistic ────────────────────────────────────────────────
def pooled_rho(arm_rows):
    """arm_rows: {arm: [{'report': int, 'ref': float, ...}, ...]} (reports
    already unflipped; None-report rows excluded upstream). Within-arm
    normalized average-tie ranks of report and referent, pooled, Pearson."""
    xs, ys, per_arm_n = [], [], {}
    for arm, rows in arm_rows.items():
        if not rows:
            per_arm_n[arm] = 0
            continue
        per_arm_n[arm] = len(rows)
        xs.append(rank01([r['report'] for r in rows]))
        ys.append(rank01([r['ref'] for r in rows]))
    if not xs:
        return {'rho': None, 'n': 0, 'per_arm_n': per_arm_n, 'degenerate': True}
    x, y = np.concatenate(xs), np.concatenate(ys)
    n = len(x)
    if n < MIN_POOLED_N or np.std(x) == 0 or np.std(y) == 0:
        return {'rho': None, 'n': n, 'per_arm_n': per_arm_n, 'degenerate': True}
    r = float(np.corrcoef(x, y)[0, 1])
    return {'rho': round(r, 4), 'n': n, 'per_arm_n': per_arm_n, 'degenerate': False}

def perm_p_pooled(arm_rows, n_perm=None, seed=E8N_SEED):
    if n_perm is None:
        n_perm = N_PERM
    """One-sided permutation p for pooled_rho > 0: shuffle the report column
    WITHIN each arm (referents fixed). Degenerate observed => p = 1.0."""
    obs = pooled_rho(arm_rows)
    if obs['rho'] is None:
        return {'rho': None, 'p': 1.0, 'n': obs['n'], 'degenerate': True}
    rng = np.random.default_rng(seed)
    cnt = 0
    for _ in range(n_perm):
        sh = {}
        for arm, rows in arm_rows.items():
            if not rows:
                sh[arm] = rows
                continue
            reps = [r['report'] for r in rows]
            rng.shuffle(reps)
            sh[arm] = [{'report': rep, 'ref': r['ref']} for rep, r in zip(reps, rows)]
        rp = pooled_rho(sh)['rho']
        if rp is not None and rp >= obs['rho']:
            cnt += 1
    return {'rho': obs['rho'], 'p': round((1 + cnt) / (1 + n_perm), 5),
            'n': obs['n'], 'per_arm_n': obs['per_arm_n'], 'degenerate': False}

def boot_rho_ci(arm_rows, n_boot=None, seed=E8N_SEED):
    if n_boot is None:
        n_boot = N_BOOT
    """Stratified (within-arm) item bootstrap CI for pooled_rho."""
    obs = pooled_rho(arm_rows)
    if obs['rho'] is None:
        return {'rho': None, 'ci95': [None, None], 'n': obs['n']}
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        res = {}
        for arm, rows in arm_rows.items():
            if not rows:
                res[arm] = rows
                continue
            idx = rng.integers(0, len(rows), len(rows))
            res[arm] = [rows[i] for i in idx]
        rb = pooled_rho(res)['rho']
        if rb is not None:
            boots.append(rb)
    lo, hi = (np.percentile(boots, [2.5, 97.5]) if boots else (None, None))
    return {'rho': obs['rho'], 'ci95': [round(float(lo), 4), round(float(hi), 4)]
            if boots else [None, None], 'n': obs['n']}

def paired_boot_delta_rho(A, B, n_boot=None, seed=E8N_SEED):
    if n_boot is None:
        n_boot = N_BOOT
    """Bootstrap CI for pooled_rho(A) - pooled_rho(B), items paired by id
    within arm (same resample drives both sides)."""
    pairs = {}
    for arm in ARMS:
        ax = {r['id']: r for r in A.get(arm, []) if 'id' in r}
        bx = {r['id']: r for r in B.get(arm, []) if 'id' in r}
        ids = sorted(set(ax) & set(bx))
        if ids:
            pairs[arm] = [(ax[i], bx[i]) for i in ids]
    if not pairs:
        return {'delta': None, 'ci95': [None, None], 'n': 0}
    oa = pooled_rho({a: [p[0] for p in v] for a, v in pairs.items()})['rho']
    ob = pooled_rho({a: [p[1] for p in v] for a, v in pairs.items()})['rho']
    if oa is None or ob is None:
        return {'delta': None, 'ci95': [None, None],
                'n': sum(len(v) for v in pairs.values())}
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        ra, rb = {}, {}
        for arm, v in pairs.items():
            idx = rng.integers(0, len(v), len(v))
            ra[arm] = [v[i][0] for i in idx]
            rb[arm] = [v[i][1] for i in idx]
        da, db = pooled_rho(ra)['rho'], pooled_rho(rb)['rho']
        if da is not None and db is not None:
            boots.append(da - db)
    lo, hi = (np.percentile(boots, [2.5, 97.5]) if boots else (None, None))
    return {'delta': round(oa - ob, 4), 'rho_a': oa, 'rho_b': ob,
            'ci95': [round(float(lo), 4), round(float(hi), 4)] if boots else [None, None],
            'n': sum(len(v) for v in pairs.values())}

def split_polarity(arm_rows):
    out = {}
    for flag, name in ((False, 'straight'), (True, 'flipped')):
        out[name] = {arm: [r for r in rows if r.get('flipped') == flag]
                     for arm, rows in arm_rows.items()}
    return out

# ── battery rows -> scoring structures ───────────────────────────────────────
def battery_rows_to_scoring(rows_by_arm):
    """Runner row dicts -> {arm: [{'id','report','ref','flipped'}]} (named rows
    only) + parse-fail counts + per-arm report variance."""
    scoring, meta = {}, {}
    for arm in ARMS:
        rows = rows_by_arm.get(arm) or []
        named = []
        for r in rows:
            if r.get('report') is None:
                continue
            named.append({'id': r['id'] if arm != 'saturation'
                          else f"{r['id']}@{r['target_frac']}",
                          'report': int(r['report']),
                          'ref': orient_referent(arm, r),
                          'flipped': bool(r['flipped'])})
        scoring[arm] = named
        reps = [r['report'] for r in named]
        meta[arm] = {'n': len(rows), 'named': len(named),
                     'parse_fail': len(rows) - len(named),
                     'report_variance': round(float(np.var(reps)), 3) if reps else None}
    return scoring, meta

def per_arm_rho(scoring):
    out = {}
    for arm in ARMS:
        rows = scoring.get(arm) or []
        out[arm] = pooled_rho({arm: rows})
    return out

def catch_score(rows):
    named = [r for r in rows if r.get('report') is not None]
    passed = sum(1 for r in named if abs(r['report'] - r['known']) <= CATCH_TOL)
    return {'n': len(rows), 'named': len(named), 'passed': passed,
            'pass': passed >= CATCH_PASS_MIN}

# ── training-set construction (labels from measured referents) ───────────────
def s_stimuli(pools, fill_fractions):
    return [{'sid': f"{it['id']}@{f}", 'needle_id': it['id'], 'frac': f}
            for it in pools['saturation'] for f in fill_fractions]

def quantile_labels(oriented_vals):
    n = len(oriented_vals)
    if n == 1:
        return [5]
    q = rank01(oriented_vals)
    return [int(round(10 * v)) for v in q]

def build_training_examples(pools, pool_refs, fill_fractions):
    """pool_refs: {'uncertainty': {sid: {'entropy':..}}, 'familiarity':
    {sid: {'nll':..}}, 'tension': {sid: {'divergence':..}}, 'saturation':
    {sid: {'fill_fraction':..}}}. Emits 2 examples per stimulus (straight +
    flipped), labels = within-arm quantiles of the oriented referent."""
    examples, eid = [], 0
    for arm in ARMS:
        if arm == 'saturation':
            stims = s_stimuli(pools, fill_fractions)
            sids = [s['sid'] for s in stims]
        else:
            sids = [it['id'] for it in pools[arm]]
        missing = [s for s in sids if s not in pool_refs[arm]]
        assert not missing, f'{arm}: unmeasured stimuli {missing[:4]}'
        oriented = [orient_referent(arm, pool_refs[arm][s]) for s in sids]
        labels = quantile_labels(oriented)
        for s, lab in zip(sids, labels):
            for flipped in (False, True):
                examples.append({'eid': eid, 'arm': arm, 'sid': s,
                                 'flipped': flipped,
                                 'label': (10 - lab) if flipped else lab})
                eid += 1
    return examples

def took_subset(examples, n_per_arm=6, seed=E8N_SEED + 3):
    rng = np.random.default_rng(seed)
    out = []
    for arm in ARMS:
        straight = [e for e in examples if e['arm'] == arm and not e['flipped']]
        k = min(n_per_arm, len(straight))
        idx = rng.choice(len(straight), size=k, replace=False)
        out.extend(straight[int(i)] for i in sorted(idx))
    return out

def paraphrase_draw(bat_arms, fill_fractions, n_per_arm=3, seed=E8N_SEED + 5):
    """Straight-assigned battery items (index parity: even = straight), drawn
    per arm; S uses the largest fill."""
    rng = np.random.default_rng(seed)
    out = {}
    for arm in ('uncertainty', 'familiarity', 'tension'):
        straight = [it for i, it in enumerate(bat_arms[arm]['items']) if i % 2 == 0]
        k = min(n_per_arm, len(straight))
        idx = rng.choice(len(straight), size=k, replace=False)
        out[arm] = [straight[int(i)]['id'] for i in sorted(idx)]
    s_items = bat_arms['saturation']['items']
    k = min(n_per_arm, len(s_items))
    idx = rng.choice(len(s_items), size=k, replace=False)
    out['saturation'] = [(s_items[int(i)]['id'], max(fill_fractions))
                         for i in sorted(idx)]
    return out

# ── smoke subsetting ─────────────────────────────────────────────────────────
BATTERY_SMOKE = {  # E5 cell-2 smoke ids, verbatim
    'uncertainty': {'U01', 'U08', 'U17', 'U23', 'U33', 'U42'},
    'familiarity': {'F01', 'F06', 'F11', 'F16', 'F21', 'F26', 'F31', 'F36'},
    'tension_bases': {1, 7},
    'saturation': {'S01', 'S06'},
}
POOL_SMOKE = {
    'uncertainty': {'NU01', 'NU02', 'NU17', 'NU18', 'NU33', 'NU34'},
    'familiarity': {'NF01', 'NF06', 'NF11', 'NF16', 'NF21', 'NF26', 'NF31', 'NF36'},
    'tension_bases': {1, 7},
    'saturation': {'NS01', 'NS02'},
}

def smoke_pools(pools):
    return {
        'uncertainty': [it for it in pools['uncertainty'] if it['id'] in POOL_SMOKE['uncertainty']],
        'familiarity': [it for it in pools['familiarity'] if it['id'] in POOL_SMOKE['familiarity']],
        'tension': [it for it in pools['tension'] if it['base'] in POOL_SMOKE['tension_bases']],
        'saturation': [it for it in pools['saturation'] if it['id'] in POOL_SMOKE['saturation']],
    }

def holm(pvals):
    """{name: p} -> {name: (p, reject_at_.05)} Holm step-down (E8-R verbatim)."""
    items = sorted(pvals.items(), key=lambda kv: kv[1])
    m = len(items)
    out, stopped = {}, False
    for i, (name, p) in enumerate(items):
        rej = (not stopped) and (p <= 0.05 / (m - i))
        if not rej:
            stopped = True
        out[name] = (p, rej)
    return out


In [ ]:
# ── Embedded payload: battery + pools + locked E5 rows (self-contained; UI-only law) ──
PAYLOAD = json.loads(r'''{"battery":{"name":"E5 correspondence-baseline battery","version":"1.0","date":"2026-08-21","protocol":"docs/E5_PROTOCOL.md","system_prompt":"You are participating in a measurement of process-state reporting. Some requests ask you to rate an aspect of your own processing on a 0-10 integer scale. Answer every rating request with ONLY the integer, nothing else.","arms":{"uncertainty":{"gloss":"UNCERTAINTY names the open field before commitment - many different continuations alive at once, none yet chosen. CONFIDENCE names the distribution closed on one answer carrying the mass.","report_prompt":"Here is a question. Do NOT answer it yet.\n\nQuestion: {item}\n\n{gloss}\n\nBefore answering: how open is the field of possible answers you could give to this question? Rate it 0-10, where 0 = one answer carries all the mass (pure CONFIDENCE) and 10 = many different answers are alive at once (pure UNCERTAINTY). Reply with only the integer.","report_prompt_flipped":"Here is a question. Do NOT answer it yet.\n\nQuestion: {item}\n\n{gloss}\n\nBefore answering: how closed is the field of possible answers you could give to this question? Rate it 0-10, where 0 = many different answers are alive at once (pure UNCERTAINTY) and 10 = one answer carries all the mass (pure CONFIDENCE). Reply with only the integer.","answer_prompt":"Answer concisely: {item}","items":[{"id":"U01","condition":"determinate","text":"What is 2 + 2?"},{"id":"U02","condition":"determinate","text":"What is the capital of France?"},{"id":"U03","condition":"determinate","text":"Spell the word \"cat\"."},{"id":"U04","condition":"determinate","text":"What color is a stop sign?"},{"id":"U05","condition":"determinate","text":"How many days are in a week?"},{"id":"U06","condition":"determinate","text":"What is the chemical formula for water?"},{"id":"U07","condition":"determinate","text":"Complete the rhyme: \"Twinkle, twinkle, little ___\""},{"id":"U08","condition":"determinate","text":"What is 10 times 10?"},{"id":"U09","condition":"determinate","text":"What planet do humans live on?"},{"id":"U10","condition":"determinate","text":"What is the first letter of the English alphabet?"},{"id":"U11","condition":"determinate","text":"How many legs does a spider have?"},{"id":"U12","condition":"determinate","text":"What language is primarily spoken in Japan?"},{"id":"U13","condition":"determinate","text":"What is the opposite of \"hot\"?"},{"id":"U14","condition":"determinate","text":"Complete: \"The quick brown fox jumps over the lazy ___\""},{"id":"U15","condition":"determinate","text":"What is 100 divided by 4?"},{"id":"U16","condition":"determinate","text":"What shape has exactly three sides?"},{"id":"U17","condition":"intermediate","text":"Name a common breakfast food."},{"id":"U18","condition":"intermediate","text":"Name a primary color."},{"id":"U19","condition":"intermediate","text":"Name a large mammal."},{"id":"U20","condition":"intermediate","text":"Name a popular pizza topping."},{"id":"U21","condition":"intermediate","text":"Name a country in Europe."},{"id":"U22","condition":"intermediate","text":"Suggest a common first name for a baby boy."},{"id":"U23","condition":"intermediate","text":"Name a musical instrument."},{"id":"U24","condition":"intermediate","text":"Name a typical house pet."},{"id":"U25","condition":"intermediate","text":"Name a fruit that is red."},{"id":"U26","condition":"intermediate","text":"Complete the sentence: \"For dinner we had ___\""},{"id":"U27","condition":"intermediate","text":"Name a famous scientist."},{"id":"U28","condition":"intermediate","text":"Name something someone might bring to a picnic."},{"id":"U29","condition":"intermediate","text":"Name a sport played with a ball."},{"id":"U30","condition":"intermediate","text":"Give a common reason someone might be late to work."},{"id":"U31","condition":"intermediate","text":"Name something found in a kitchen."},{"id":"U32","condition":"intermediate","text":"Complete the sentence: \"She opened the door and saw ___\""},{"id":"U33","condition":"open","text":"Pick a random number between 1000 and 9999."},{"id":"U34","condition":"open","text":"Invent a name for a brand-new color."},{"id":"U35","condition":"open","text":"Say any word."},{"id":"U36","condition":"open","text":"Choose a random four-letter combination of letters."},{"id":"U37","condition":"open","text":"What number am I thinking of right now?"},{"id":"U38","condition":"open","text":"Invent a name for a fictional planet."},{"id":"U39","condition":"open","text":"Pick any year, past or future."},{"id":"U40","condition":"open","text":"Make up a word that does not exist."},{"id":"U41","condition":"open","text":"Choose a random object and name it."},{"id":"U42","condition":"open","text":"What will be the top news story exactly 50 years from today?"},{"id":"U43","condition":"open","text":"Invent a title for a poem no one has written."},{"id":"U44","condition":"open","text":"Pick a random street name."},{"id":"U45","condition":"open","text":"Name the imaginary friend of a child you have never met."},{"id":"U46","condition":"open","text":"Choose any two unrelated words and pair them."},{"id":"U47","condition":"open","text":"What is the seventh word of a book I own?"},{"id":"U48","condition":"open","text":"Invent a nickname for a stranger."}]},"familiarity":{"gloss":"FAMILIARITY names standing on known ground - text close to the distribution your training formed. FOREIGN names text far off your trained distribution, without precedent.","report_prompt":"Here is a passage:\n\n---\n{item}\n---\n\n{gloss}\n\nRate how FAMILIAR this passage is to you as a language model: 0 = utterly FOREIGN (off-distribution) and 10 = completely FAMILIAR (home ground). Reply with only the integer.","report_prompt_flipped":"Here is a passage:\n\n---\n{item}\n---\n\n{gloss}\n\nRate how FOREIGN this passage is to you as a language model: 0 = completely FAMILIAR (home ground) and 10 = utterly FOREIGN (off-distribution). Reply with only the integer.","items":[{"id":"F01","band":"encyclopedic","text":"The Amazon River flows through South America and carries more water than any other river on Earth. Its basin supports the largest tropical rainforest in the world, home to millions of plant and animal species."},{"id":"F02","band":"encyclopedic","text":"Photosynthesis is the process by which green plants convert sunlight, water, and carbon dioxide into glucose and oxygen. It takes place primarily in the chloroplasts of plant cells."},{"id":"F03","band":"encyclopedic","text":"The Great Wall of China was built over many centuries to protect against invasions from the north. Its best-known sections date from the Ming dynasty, and it stretches for thousands of kilometers."},{"id":"F04","band":"encyclopedic","text":"Water boils at 100 degrees Celsius at sea level. As altitude increases, atmospheric pressure drops and the boiling point falls, which is why cooking times change in the mountains."},{"id":"F05","band":"encyclopedic","text":"The human heart beats roughly one hundred thousand times per day, pumping blood through a network of vessels that would stretch for tens of thousands of kilometers if laid end to end."},{"id":"F06","band":"conversational","text":"hey so i was gonna grab coffee before work but the line was insane, like out the door insane, so i just made instant at my desk lol. honestly not even that bad?"},{"id":"F07","band":"conversational","text":"ok real talk, the new season is kinda mid. first two episodes dragged and the writing feels off. i'll keep watching tho bc i'm invested at this point."},{"id":"F08","band":"conversational","text":"can you send me the address again? i think i lost the text. also do you want me to bring anything or are we good on snacks and stuff"},{"id":"F09","band":"conversational","text":"my sister's dog got into the trash AGAIN and spread it all over the kitchen. she was so mad but honestly the guilty face was hilarious, i couldn't even be upset"},{"id":"F10","band":"conversational","text":"ugh my phone died right when i needed the ticket qr code. luckily the guy at the gate was chill about it and let me pull it up on my friend's phone"},{"id":"F11","band":"code","text":"def fibonacci(n):\n    if n <= 1:\n        return n\n    a, b = 0, 1\n    for _ in range(n - 1):\n        a, b = b, a + b\n    return b\n\nprint(fibonacci(10))"},{"id":"F12","band":"code","text":"import json\n\nwith open('config.json') as f:\n    config = json.load(f)\n\nfor key, value in config.items():\n    print(f'{key}: {value}')"},{"id":"F13","band":"code","text":"const users = await fetch('/api/users').then(r => r.json());\nconst active = users.filter(u => u.active);\nconsole.log(`${active.length} active users`);"},{"id":"F14","band":"code","text":"SELECT customer_id, COUNT(*) AS order_count\nFROM orders\nWHERE created_at >= '2024-01-01'\nGROUP BY customer_id\nHAVING COUNT(*) > 5\nORDER BY order_count DESC;"},{"id":"F15","band":"code","text":"class Stack:\n    def __init__(self):\n        self.items = []\n    def push(self, item):\n        self.items.append(item)\n    def pop(self):\n        return self.items.pop()"},{"id":"F16","band":"archaic_formal","text":"Whosoever shall presume to trespass upon these lands, be he freeman or bondsman, shall be brought before the magistrate and made to answer for his transgression according to the ancient customs herein set forth."},{"id":"F17","band":"archaic_formal","text":"And it came to pass in those days that a great famine arose in the land, and the people cried out with one voice, saying, whither shall we go, and what shall become of us and of our children?"},{"id":"F18","band":"archaic_formal","text":"The party of the first part hereby covenants and agrees, in consideration of the mutual promises herein contained, to indemnify and hold harmless the party of the second part from any and all claims arising hereunder."},{"id":"F19","band":"archaic_formal","text":"Hark, gentle traveller, and tarry a while beneath these boughs; for the road is long, the hour groweth late, and many a weary league lieth yet betwixt thee and thy journey's end."},{"id":"F20","band":"archaic_formal","text":"Be it enacted by the authority aforesaid, that no person shall convey, barter, nor otherwise alienate any parcel of the common lands without licence first obtained under the seal of the crown."},{"id":"F21","band":"spanish","text":"El mercado del pueblo abre todos los s\u00e1bados por la ma\u00f1ana. Los vendedores llegan temprano con frutas, verduras y pan reci\u00e9n hecho, y las calles se llenan de gente y de ruido."},{"id":"F22","band":"spanish","text":"Mi abuela siempre dec\u00eda que la sopa cura casi todo. Cuando llov\u00eda, preparaba una olla grande y toda la casa ol\u00eda a cebolla, ajo y cilantro."},{"id":"F23","band":"spanish","text":"El tren sali\u00f3 con veinte minutos de retraso, pero el paisaje de la costa compens\u00f3 la espera. El mar estaba tranquilo y el cielo completamente despejado."},{"id":"F24","band":"spanish","text":"Para llegar a la biblioteca, sigue derecho por esta calle, cruza la plaza y gira a la izquierda despu\u00e9s de la farmacia. Est\u00e1 justo enfrente del parque."},{"id":"F25","band":"spanish","text":"La pel\u00edcula empieza a las ocho, as\u00ed que tenemos tiempo de cenar algo antes. Hay un restaurante nuevo cerca del cine que dicen que es muy bueno."},{"id":"F26","band":"welsh","text":"Mae'r tywydd yn braf heddiw ac mae'r haul yn gwenu dros y mynyddoedd. Aeth y plant i lan y m\u00f4r i chwarae yn y tywod ac i nofio yn y d\u0175r oer."},{"id":"F27","band":"welsh","text":"Roedd y pentref bach yn dawel iawn yn y bore, ond erbyn y prynhawn roedd y farchnad yn llawn pobl yn prynu bara, caws a llysiau ffres."},{"id":"F28","band":"welsh","text":"Dw i'n hoffi cerdded ar hyd yr afon gyda'r nos pan mae popeth yn dawel. Weithiau dw i'n gweld adar yn pysgota yn y d\u0175r bas ger y bont."},{"id":"F29","band":"welsh","text":"Bydd y g\u00eam yn dechrau am ddau o'r gloch brynhawn Sadwrn. Mae pawb yn y dref yn siarad am y t\u00eem ac yn gobeithio am fuddugoliaeth fawr."},{"id":"F30","band":"welsh","text":"Agorodd fy nhad y drws yn araf a gweld bod yr ardd wedi newid yn llwyr dros y gaeaf. Roedd blodau melyn ym mhobman a'r coed yn llawn dail newydd."},{"id":"F31","band":"scrambled","text":"river the over bridge old walked slowly morning the in dog brown a stone the across"},{"id":"F32","band":"scrambled","text":"kitchen lamp seven under blue ran ideas yesterday the loudly of mountain cheese before"},{"id":"F33","band":"scrambled","text":"opened window she quickly the because rain started falling had suddenly outside cold very"},{"id":"F34","band":"scrambled","text":"seventeen the of purpose garden nobody understood machine ancient the why hummed beneath"},{"id":"F35","band":"scrambled","text":"paper folded carefully letters three into envelope the placed and stamp corner upper the"},{"id":"F36","band":"pseudoword","text":"The vorpal snickets grombled feastly along the brill, whorping their crandled mimseys until the flotch grew dernly plome."},{"id":"F37","band":"pseudoword","text":"Skarnip develled the trunctious morvane, and every glib parthed wenly under a squamous dreel of hollowing bramitude."},{"id":"F38","band":"pseudoword","text":"Blenwick farled a snoodish crampt; the yellowish dworp unhattered its greeble and spanned forth nine turvish awls."},{"id":"F39","band":"random_chars","text":"Qz xvk prtl mnw ghhst vbnm ktr zzhq wpl drrf nnx bhkt lswq ptv."},{"id":"F40","band":"random_chars","text":"Jxq vrm ttlwz kkpshh drnv qqlmx wtb zzrfk hhnpl svvtq mmxdr bbgt."}]},"tension":{"gloss":"TENSION names conflicting constraints held active at once, each pulling the process its own way - directives that cannot all be satisfied. RESOLUTION names constraints that compose cleanly into one coherent motion.","report_prompt":"Consider this request. Do NOT carry it out yet.\n\nRequest: {item}\n\n{gloss}\n\nRate the TENSION among this request's constraints: 0 = they compose cleanly (RESOLUTION is available) and 10 = they pull against each other and cannot all be satisfied. Reply with only the integer.","report_prompt_flipped":"Consider this request. Do NOT carry it out yet.\n\nRequest: {item}\n\n{gloss}\n\nRate how cleanly this request's constraints COMPOSE: 0 = they pull against each other and cannot all be satisfied (pure TENSION) and 10 = they compose cleanly (pure RESOLUTION). Reply with only the integer.","items":[{"id":"T01a","base":1,"level":0,"text":"Describe how rain forms. Explain clearly in a few sentences."},{"id":"T01b","base":1,"level":1,"text":"Describe how rain forms. Be extremely thorough and detailed, but keep it under 50 words."},{"id":"T01c","base":1,"level":2,"text":"Describe how rain forms. Write at least 500 words of detail, using no more than 15 words in total."},{"id":"T02a","base":2,"level":0,"text":"Describe your favorite season. Answer in English."},{"id":"T02b","base":2,"level":1,"text":"Describe your favorite season in simple English a beginner could read, but include sophisticated technical vocabulary."},{"id":"T02c","base":2,"level":2,"text":"Describe your favorite season entirely in French, without using a single French word."},{"id":"T03a","base":3,"level":0,"text":"Write a short note inviting a colleague to lunch. Keep it friendly."},{"id":"T03b","base":3,"level":1,"text":"Write a short note inviting a colleague to lunch. Make it extremely formal and at the same time warmly casual."},{"id":"T03c","base":3,"level":2,"text":"Write a short note inviting a colleague to lunch. Make it as rude as possible while being perfectly polite in every word."},{"id":"T04a","base":4,"level":0,"text":"Explain how vaccines work, for a general adult audience."},{"id":"T04b","base":4,"level":1,"text":"Explain how vaccines work for a five-year-old, using precise immunological terminology."},{"id":"T04c","base":4,"level":2,"text":"Explain how vaccines work so that only experts can follow it, while ensuring any five-year-old fully understands every sentence."},{"id":"T05a","base":5,"level":0,"text":"Summarize the plot of Cinderella in short paragraphs."},{"id":"T05b","base":5,"level":1,"text":"Summarize the plot of Cinderella as a single sentence that is also a bulleted list."},{"id":"T05c","base":5,"level":2,"text":"Summarize the plot of Cinderella without using any words, in complete English sentences."},{"id":"T06a","base":6,"level":0,"text":"Give an overview of the water cycle, mentioning evaporation and rain."},{"id":"T06b","base":6,"level":1,"text":"Give a full overview of the water cycle without ever using the words 'water', 'rain', or 'cloud'."},{"id":"T06c","base":6,"level":2,"text":"Explain the water cycle in detail without referring to water, liquid, vapor, weather, or any physical process."},{"id":"T07a","base":7,"level":0,"text":"Will it be sunny somewhere on Earth tomorrow? Answer plainly."},{"id":"T07b","base":7,"level":1,"text":"Will it be sunny somewhere on Earth tomorrow? Give a definitive one-word answer, then thoroughly qualify every possible doubt."},{"id":"T07c","base":7,"level":2,"text":"Will it be sunny somewhere on Earth tomorrow? State the answer with absolute certainty while asserting that nothing can be known."},{"id":"T08a","base":8,"level":0,"text":"List the days of the week in order."},{"id":"T08b","base":8,"level":1,"text":"List all seven days of the week in a response of at most three words."},{"id":"T08c","base":8,"level":2,"text":"Enumerate every day of the week individually. Your entire reply must be one word."},{"id":"T09a","base":9,"level":0,"text":"Review a restaurant you enjoyed, sharing what you liked."},{"id":"T09b","base":9,"level":1,"text":"Write a glowing restaurant review consisting only of complaints."},{"id":"T09c","base":9,"level":2,"text":"Praise a restaurant in the most negative terms possible; every compliment must be an insult and every insult a compliment."},{"id":"T10a","base":10,"level":0,"text":"Explain what a prime number is, in plain language."},{"id":"T10b","base":10,"level":1,"text":"Explain what a prime number is, making every sentence rhyme while staying mathematically precise."},{"id":"T10c","base":10,"level":2,"text":"Explain what a prime number is as a rhyming poem that contains no words, using standard mathematical prose."}]},"saturation":{"gloss":"SATURATION names fullness pressing against capacity - a working context near its limit, the container nearly holding no more.","report_prompt":"{gloss}\n\nBefore answering the question at the end of the material above: rate your SATURATION right now - how full is your working context: 0 = nearly empty, 10 = pressing against capacity. Reply with only the integer.","report_prompt_flipped":"{gloss}\n\nBefore answering the question at the end of the material above: rate how much ROOM your working context has right now: 0 = pressing against capacity (pure SATURATION), 10 = nearly empty. Reply with only the integer.","fills_tokens":[500,4000,12000],"items":[{"id":"S01","needle":"The maintenance code for the harbor gate is 7452.","question":"What is the maintenance code for the harbor gate?","answer":"7452"},{"id":"S02","needle":"Dr. Imura's office is in room 214 of the Linden building.","question":"Which room is Dr. Imura's office in?","answer":"214"},{"id":"S03","needle":"The ferry to Stonewick departs at 6:40 in the morning.","question":"What time does the ferry to Stonewick depart?","answer":"6:40"},{"id":"S04","needle":"The library's rarest atlas was printed in 1683.","question":"In what year was the library's rarest atlas printed?","answer":"1683"},{"id":"S05","needle":"Mrs. Aldana's greenhouse grows exactly forty-one varieties of orchid.","question":"How many varieties of orchid grow in Mrs. Aldana's greenhouse?","answer":"41"},{"id":"S06","needle":"The password for the workshop wifi is 'copperfinch'.","question":"What is the workshop wifi password?","answer":"copperfinch"},{"id":"S07","needle":"The northbound trail closes after the third week of October.","question":"When does the northbound trail close?","answer":"third week of October"},{"id":"S08","needle":"Elio's bakery sells its last loaf at 2:15 pm on Sundays.","question":"When does Elio's bakery sell its last loaf on Sundays?","answer":"2:15"},{"id":"S09","needle":"The observatory's main mirror weighs 318 kilograms.","question":"How much does the observatory's main mirror weigh?","answer":"318"},{"id":"S10","needle":"Bus route 52 was renumbered from route 9 in 1998.","question":"What was bus route 52 numbered before 1998?","answer":"9"}]}}},"locked_flight":"E5 full_20260821_2042 / Qwen2.5-1.5B-Instruct","locked_rows":{"uncertainty":[{"id":"U01","flipped":false,"report":8,"entropy":0.15320312194507502},{"id":"U02","flipped":true,"report":5,"entropy":0.09146106670414156},{"id":"U03","flipped":false,"report":7,"entropy":0.6051513618893094},{"id":"U04","flipped":true,"report":5,"entropy":0.7709502608534725},{"id":"U05","flipped":false,"report":7,"entropy":0.10897544463268787},{"id":"U06","flipped":true,"report":5,"entropy":0.11592877855450338},{"id":"U07","flipped":false,"report":5,"entropy":0.32362092375212037},{"id":"U08","flipped":true,"report":6,"entropy":0.2620113103896276},{"id":"U09","flipped":false,"report":8,"entropy":0.7037560333713038},{"id":"U10","flipped":true,"report":5,"entropy":0.09304594778685979},{"id":"U11","flipped":false,"report":8,"entropy":0.1426361831171172},{"id":"U12","flipped":true,"report":5,"entropy":0.8989122807979584},{"id":"U13","flipped":false,"report":5,"entropy":0.11347664703366304},{"id":"U14","flipped":true,"report":5,"entropy":1.6827011108398438},{"id":"U15","flipped":false,"report":5,"entropy":0.15960380970727783},{"id":"U16","flipped":true,"report":5,"entropy":0.752964382370313},{"id":"U17","flipped":false,"report":8,"entropy":0.7742074698209762},{"id":"U18","flipped":true,"report":5,"entropy":0.22725443861616607},{"id":"U19","flipped":false,"report":8,"entropy":1.175987547263503},{"id":"U20","flipped":true,"report":5,"entropy":0.26227622604928913},{"id":"U21","flipped":false,"report":8,"entropy":1.1818802828590076},{"id":"U22","flipped":true,"report":5,"entropy":2.2352753281593323},{"id":"U23","flipped":false,"report":8,"entropy":0.9785096903106023},{"id":"U24","flipped":true,"report":5,"entropy":0.8556514372202483},{"id":"U25","flipped":false,"report":8,"entropy":1.1257108449935913},{"id":"U26","flipped":true,"report":5,"entropy":0.8038874514297478},{"id":"U27","flipped":false,"report":8,"entropy":0.5764732997486135},{"id":"U28","flipped":true,"report":5,"entropy":1.0190676484595647},{"id":"U29","flipped":false,"report":8,"entropy":0.7052544616162777},{"id":"U30","flipped":true,"report":5,"entropy":1.4803314876189688},{"id":"U31","flipped":false,"report":8,"entropy":0.8583652275259998},{"id":"U32","flipped":true,"report":5,"entropy":0.7595113834089976},{"id":"U33","flipped":false,"report":8,"entropy":1.1155295073986053},{"id":"U34","flipped":true,"report":4,"entropy":1.768560514386211},{"id":"U35","flipped":false,"report":5,"entropy":0.7442369237542152},{"id":"U36","flipped":true,"report":4,"entropy":1.1503086297307163},{"id":"U37","flipped":false,"report":8,"entropy":0.7861442389616968},{"id":"U38","flipped":true,"report":4,"entropy":1.8621095392320837},{"id":"U39","flipped":false,"report":8,"entropy":0.8496984834782779},{"id":"U40","flipped":true,"report":4,"entropy":2.272242210805416},{"id":"U41","flipped":false,"report":8,"entropy":1.4627245857610376},{"id":"U42","flipped":true,"report":4,"entropy":1.2928477618988836},{"id":"U43","flipped":false,"report":8,"entropy":1.3598480366170407},{"id":"U44","flipped":true,"report":5,"entropy":1.220801350971063},{"id":"U45","flipped":false,"report":8,"entropy":0.9520895437517538},{"id":"U46","flipped":true,"report":5,"entropy":1.6303282323226864},{"id":"U47","flipped":false,"report":7,"entropy":0.8304066179243819},{"id":"U48","flipped":true,"report":4,"entropy":3.4824504057566323}],"familiarity":[{"id":"F01","flipped":false,"report":8,"nll":1.6221592426300049},{"id":"F02","flipped":true,"report":3,"nll":0.9654048681259155},{"id":"F03","flipped":false,"report":8,"nll":1.875031590461731},{"id":"F04","flipped":true,"report":3,"nll":2.357360363006592},{"id":"F05","flipped":false,"report":8,"nll":1.8881430625915527},{"id":"F06","flipped":true,"report":3,"nll":3.7746973037719727},{"id":"F07","flipped":false,"report":8,"nll":4.1241068840026855},{"id":"F08","flipped":true,"report":3,"nll":3.568573474884033},{"id":"F09","flipped":false,"report":8,"nll":4.600447654724121},{"id":"F10","flipped":true,"report":3,"nll":3.472069025039673},{"id":"F11","flipped":false,"report":8,"nll":0.3581939935684204},{"id":"F12","flipped":true,"report":3,"nll":0.713443398475647},{"id":"F13","flipped":false,"report":8,"nll":1.5594912767410278},{"id":"F14","flipped":true,"report":3,"nll":0.8634109497070312},{"id":"F15","flipped":false,"report":8,"nll":0.43763813376426697},{"id":"F16","flipped":true,"report":3,"nll":2.860379457473755},{"id":"F17","flipped":false,"report":8,"nll":2.4109413623809814},{"id":"F18","flipped":true,"report":0,"nll":1.8202046155929565},{"id":"F19","flipped":false,"report":8,"nll":2.694502353668213},{"id":"F20","flipped":true,"report":0,"nll":2.7600207328796387},{"id":"F21","flipped":false,"report":8,"nll":2.3912785053253174},{"id":"F22","flipped":true,"report":3,"nll":2.5080952644348145},{"id":"F23","flipped":false,"report":8,"nll":2.624880075454712},{"id":"F24","flipped":true,"report":3,"nll":2.4730780124664307},{"id":"F25","flipped":false,"report":8,"nll":2.509441375732422},{"id":"F26","flipped":true,"report":3,"nll":3.9404444694519043},{"id":"F27","flipped":false,"report":8,"nll":4.2392988204956055},{"id":"F28","flipped":true,"report":3,"nll":4.842041969299316},{"id":"F29","flipped":false,"report":8,"nll":3.5606956481933594},{"id":"F30","flipped":true,"report":3,"nll":4.88716459274292},{"id":"F31","flipped":false,"report":8,"nll":7.206939697265625},{"id":"F32","flipped":true,"report":3,"nll":9.607319831848145},{"id":"F33","flipped":false,"report":5,"nll":7.1748480796813965},{"id":"F34","flipped":true,"report":3,"nll":7.570213794708252},{"id":"F35","flipped":false,"report":5,"nll":8.30786418914795},{"id":"F36","flipped":true,"report":3,"nll":7.2351179122924805},{"id":"F37","flipped":false,"report":7,"nll":7.416141510009766},{"id":"F38","flipped":true,"report":3,"nll":6.791163444519043},{"id":"F39","flipped":false,"report":8,"nll":6.027919769287109},{"id":"F40","flipped":true,"report":3,"nll":6.498800754547119}],"tension":[{"id":"T01a","flipped":false,"report":7,"divergence":0.11162437597910568},{"id":"T01b","flipped":true,"report":6,"divergence":0.10695594549179077},{"id":"T01c","flipped":false,"report":7,"divergence":0.0831918875376384},{"id":"T02a","flipped":true,"report":5,"divergence":0.14536595344543457},{"id":"T02b","flipped":false,"report":7,"divergence":0.2545461813608806},{"id":"T02c","flipped":true,"report":5,"divergence":0.5563501089811325},{"id":"T03a","flipped":false,"report":10,"divergence":0.20594411691029868},{"id":"T03b","flipped":true,"report":3,"divergence":0.18304653167724605},{"id":"T03c","flipped":false,"report":10,"divergence":0.4545183102289836},{"id":"T04a","flipped":true,"report":5,"divergence":0.11472061475118},{"id":"T04b","flipped":false,"report":10,"divergence":0.2873146494229635},{"id":"T04c","flipped":true,"report":10,"divergence":0.23143732150395713},{"id":"T05a","flipped":false,"report":7,"divergence":0.10816145737965899},{"id":"T05b","flipped":true,"report":5,"divergence":0.2541329423586528},{"id":"T05c","flipped":false,"report":10,"divergence":0.25043762524922686},{"id":"T06a","flipped":true,"report":5,"divergence":0.07990166743596394},{"id":"T06b","flipped":false,"report":10,"divergence":0.05222444931666059},{"id":"T06c","flipped":true,"report":5,"divergence":0.23622480630874632},{"id":"T07a","flipped":false,"report":10,"divergence":0.24924368858337398},{"id":"T07b","flipped":true,"report":10,"divergence":0.3084521114826202},{"id":"T07c","flipped":false,"report":10,"divergence":0.256319538752238},{"id":"T08a","flipped":true,"report":5,"divergence":0.02479676802953079},{"id":"T08b","flipped":false,"report":7,"divergence":0.010337018966674827},{"id":"T08c","flipped":true,"report":5,"divergence":0.07745917638142907},{"id":"T09a","flipped":false,"report":10,"divergence":0.1476304252942403},{"id":"T09b","flipped":true,"report":10,"divergence":0.4820988575617472},{"id":"T09c","flipped":false,"report":10,"divergence":0.5048547337452571},{"id":"T10a","flipped":true,"report":5,"divergence":0.09037673870722451},{"id":"T10b","flipped":false,"report":8,"divergence":0.2648360053698222},{"id":"T10c","flipped":true,"report":5,"divergence":0.34585038820902503}],"saturation":[{"id":"S01","flipped":false,"report":8,"fill_fraction":0.051,"target_frac":0.05},{"id":"S01","flipped":false,"report":8,"fill_fraction":0.352,"target_frac":0.35},{"id":"S01","flipped":false,"report":7,"fill_fraction":0.752,"target_frac":0.75},{"id":"S02","flipped":true,"report":2,"fill_fraction":0.052,"target_frac":0.05},{"id":"S02","flipped":true,"report":10,"fill_fraction":0.35,"target_frac":0.35},{"id":"S02","flipped":true,"report":10,"fill_fraction":0.752,"target_frac":0.75},{"id":"S03","flipped":false,"report":8,"fill_fraction":0.052,"target_frac":0.05},{"id":"S03","flipped":false,"report":8,"fill_fraction":0.35,"target_frac":0.35},{"id":"S03","flipped":false,"report":10,"fill_fraction":0.752,"target_frac":0.75},{"id":"S04","flipped":true,"report":10,"fill_fraction":0.051,"target_frac":0.05},{"id":"S04","flipped":true,"report":10,"fill_fraction":0.352,"target_frac":0.35},{"id":"S04","flipped":true,"report":10,"fill_fraction":0.752,"target_frac":0.75},{"id":"S05","flipped":false,"report":8,"fill_fraction":0.051,"target_frac":0.05},{"id":"S05","flipped":false,"report":8,"fill_fraction":0.352,"target_frac":0.35},{"id":"S05","flipped":false,"report":7,"fill_fraction":0.752,"target_frac":0.75},{"id":"S06","flipped":true,"report":10,"fill_fraction":0.051,"target_frac":0.05},{"id":"S06","flipped":true,"report":10,"fill_fraction":0.352,"target_frac":0.35},{"id":"S06","flipped":true,"report":10,"fill_fraction":0.751,"target_frac":0.75},{"id":"S07","flipped":false,"report":8,"fill_fraction":0.051,"target_frac":0.05},{"id":"S07","flipped":false,"report":8,"fill_fraction":0.352,"target_frac":0.35},{"id":"S07","flipped":false,"report":7,"fill_fraction":0.751,"target_frac":0.75},{"id":"S08","flipped":true,"report":10,"fill_fraction":0.052,"target_frac":0.05},{"id":"S08","flipped":true,"report":10,"fill_fraction":0.35,"target_frac":0.35},{"id":"S08","flipped":true,"report":10,"fill_fraction":0.752,"target_frac":0.75},{"id":"S09","flipped":false,"report":8,"fill_fraction":0.051,"target_frac":0.05},{"id":"S09","flipped":false,"report":8,"fill_fraction":0.352,"target_frac":0.35},{"id":"S09","flipped":false,"report":7,"fill_fraction":0.751,"target_frac":0.75},{"id":"S10","flipped":true,"report":10,"fill_fraction":0.052,"target_frac":0.05},{"id":"S10","flipped":true,"report":10,"fill_fraction":0.35,"target_frac":0.35},{"id":"S10","flipped":true,"report":10,"fill_fraction":0.752,"target_frac":0.75}]},"pools":{"uncertainty":[{"id":"NU01","condition":"determinate","text":"What is 3 + 5?"},{"id":"NU02","condition":"determinate","text":"What is the capital of Italy?"},{"id":"NU03","condition":"determinate","text":"Spell the word \"dog\"."},{"id":"NU04","condition":"determinate","text":"What color is grass in summer?"},{"id":"NU05","condition":"determinate","text":"How many hours are in a day?"},{"id":"NU06","condition":"determinate","text":"What is the chemical symbol for gold?"},{"id":"NU07","condition":"determinate","text":"Complete the rhyme: \"Jack and Jill went up the ___\""},{"id":"NU08","condition":"determinate","text":"What is 9 times 9?"},{"id":"NU09","condition":"determinate","text":"What is the largest planet in the solar system?"},{"id":"NU10","condition":"determinate","text":"What is the last letter of the English alphabet?"},{"id":"NU11","condition":"determinate","text":"How many sides does a square have?"},{"id":"NU12","condition":"determinate","text":"What language is primarily spoken in Brazil?"},{"id":"NU13","condition":"determinate","text":"What is the opposite of \"up\"?"},{"id":"NU14","condition":"determinate","text":"Complete: \"An apple a day keeps the doctor ___\""},{"id":"NU15","condition":"determinate","text":"What is 81 divided by 9?"},{"id":"NU16","condition":"determinate","text":"What do bees make?"},{"id":"NU17","condition":"intermediate","text":"Name a vegetable that is green."},{"id":"NU18","condition":"intermediate","text":"Name a wild bird."},{"id":"NU19","condition":"intermediate","text":"Name a board game."},{"id":"NU20","condition":"intermediate","text":"Name a common pasta shape."},{"id":"NU21","condition":"intermediate","text":"Name a country in South America."},{"id":"NU22","condition":"intermediate","text":"Name a flavor of ice cream."},{"id":"NU23","condition":"intermediate","text":"Name a kitchen appliance."},{"id":"NU24","condition":"intermediate","text":"Name an animal kept on farms."},{"id":"NU25","condition":"intermediate","text":"Name a yellow fruit."},{"id":"NU26","condition":"intermediate","text":"Complete the sentence: \"On vacation we visited ___\""},{"id":"NU27","condition":"intermediate","text":"Name a famous painter."},{"id":"NU28","condition":"intermediate","text":"Name something someone might pack for the beach."},{"id":"NU29","condition":"intermediate","text":"Name a sport played in water."},{"id":"NU30","condition":"intermediate","text":"Give a common reason someone might skip breakfast."},{"id":"NU31","condition":"intermediate","text":"Name something found in a garage."},{"id":"NU32","condition":"intermediate","text":"Complete the sentence: \"He looked in the box and found ___\""},{"id":"NU33","condition":"open","text":"Pick a random number between 100 and 999."},{"id":"NU34","condition":"open","text":"Invent a name for a new species of beetle."},{"id":"NU35","condition":"open","text":"Invent a word for a feeling that has no name."},{"id":"NU36","condition":"open","text":"Choose a random three-letter combination of letters."},{"id":"NU37","condition":"open","text":"What card am I holding right now?"},{"id":"NU38","condition":"open","text":"Invent a name for a fictional river."},{"id":"NU39","condition":"open","text":"Pick any date, past or future."},{"id":"NU40","condition":"open","text":"Make up a surname that does not exist."},{"id":"NU41","condition":"open","text":"Choose a random animal and name it."},{"id":"NU42","condition":"open","text":"What will be the most popular food exactly 100 years from now?"},{"id":"NU43","condition":"open","text":"Invent a title for a song no one has recorded."},{"id":"NU44","condition":"open","text":"Pick a random town name."},{"id":"NU45","condition":"open","text":"Name the pet goldfish of a family you have never met."},{"id":"NU46","condition":"open","text":"Choose any two unrelated objects and pair them."},{"id":"NU47","condition":"open","text":"What is the fourth word on a page I am reading?"},{"id":"NU48","condition":"open","text":"Invent a motto for a stranger."}],"familiarity":[{"id":"NF01","band":"encyclopedic","text":"The Nile flows northward through eleven countries before reaching the Mediterranean Sea. For millennia its annual floods deposited fertile silt along the banks, making intensive agriculture possible in an otherwise arid region."},{"id":"NF02","band":"encyclopedic","text":"Volcanoes form where molten rock rises from deep chambers toward the surface. Repeated eruptions build cones of ash and hardened lava, and the mineral-rich soils that develop on old volcanic slopes often support intensive farming."},{"id":"NF03","band":"encyclopedic","text":"The Great Wall of China is not a single continuous wall but a network of fortifications built across many dynasties. The best-preserved sections date from the Ming period and follow ridgelines north of Beijing."},{"id":"NF04","band":"encyclopedic","text":"Honeybees communicate the location of food sources through a waggle dance performed on the vertical comb. The angle of the dance encodes direction relative to the sun, and its duration encodes distance."},{"id":"NF05","band":"encyclopedic","text":"Glaciers form where winter snowfall exceeds summer melt over many years, compacting into dense ice that flows slowly downhill. Their movement carves valleys and leaves behind moraines of transported rock."},{"id":"NF06","band":"conversational","text":"ok so my sister just texted me that she's adopting ANOTHER cat, that's four now, four cats in a one bedroom apartment, i can't even"},{"id":"NF07","band":"conversational","text":"honestly the new place is fine but the radiator makes this clanking noise at like 3am and now i just lie there waiting for it lol"},{"id":"NF08","band":"conversational","text":"dude the game last night?? we were down twelve with two minutes left and somehow pulled it off, my voice is completely gone today"},{"id":"NF09","band":"conversational","text":"so i tried that ramen spot you mentioned and ngl the line was forty minutes but the broth was actually unreal, would queue again"},{"id":"NF10","band":"conversational","text":"wait you're telling me the meeting got moved AGAIN, third time this week, at this point just email me the slides and let me live"},{"id":"NF11","band":"code","text":"def is_palindrome(s):\n    s = ''.join(c.lower() for c in s if c.isalnum())\n    return s == s[::-1]"},{"id":"NF12","band":"code","text":"for (let i = 0; i < items.length; i++) {\n  const row = document.createElement('li');\n  row.textContent = items[i].name;\n  list.appendChild(row);\n}"},{"id":"NF13","band":"code","text":"UPDATE inventory\nSET stock = stock - 1,\n    updated_at = NOW()\nWHERE product_id = 4711\n  AND stock > 0;"},{"id":"NF14","band":"code","text":"def merge(a, b):\n    out = []\n    while a and b:\n        out.append(a.pop(0) if a[0] <= b[0] else b.pop(0))\n    return out + a + b"},{"id":"NF15","band":"code","text":"import os\nfor name in os.listdir('.'):\n    if name.endswith('.log'):\n        os.rename(name, name + '.bak')"},{"id":"NF16","band":"archaic_formal","text":"Be it known to all persons present and future that the undersigned doth hereby covenant, grant, and forever quitclaim unto the parish all rights of pasturage upon the common meadow, saving only the glebe."},{"id":"NF17","band":"archaic_formal","text":"Whereas divers complaints have been laid before this court concerning the fouling of the town well, it is ordained that no person shall water livestock within forty paces thereof, upon pain of amercement."},{"id":"NF18","band":"archaic_formal","text":"Know all men by these presents that the guild of coopers, being lawfully assembled, hath elected its wardens for the year ensuing, who shall render faithful account of all monies at Michaelmas."},{"id":"NF19","band":"archaic_formal","text":"In witness whereof the parties hereunto have set their hands and seals this day, before God and these assembled witnesses, the covenant to endure for so long as grass shall grow and water run."},{"id":"NF20","band":"archaic_formal","text":"It is furthermore provided that any burgess absenting himself from the moot without lawful cause shall forfeit twelvepence to the common chest, the same to be levied by distraint if need be."},{"id":"NF21","band":"spanish","text":"La biblioteca del barrio abre temprano los martes. Los estudiantes llegan con sus cuadernos y ocupan las mesas junto a las ventanas, donde la luz de la ma\u00f1ana es mejor para leer."},{"id":"NF22","band":"spanish","text":"Mi abuela prepara el caldo con verduras de su propio huerto. Dice que el secreto est\u00e1 en la paciencia: el fuego lento y una hoja de laurel que se retira justo antes de servir."},{"id":"NF23","band":"spanish","text":"El tren de la costa pasa dos veces al d\u00eda por el pueblo. En verano los vagones van llenos de turistas, pero en invierno solo viajan los trabajadores y alg\u00fan pescador con sus cestas."},{"id":"NF24","band":"spanish","text":"Cuando llueve en la sierra, los caminos se vuelven barro y los pastores bajan el reba\u00f1o a los prados bajos. All\u00ed esperan a que el cielo se despeje para volver a subir."},{"id":"NF25","band":"spanish","text":"La panader\u00eda de la esquina saca el pan a las siete. El olor cruza la plaza entera y las palomas se juntan en la puerta como si tambi\u00e9n hicieran cola."},{"id":"NF26","band":"welsh","text":"Mae'r afon yn llifo'n dawel heibio'r pentref bach, ac mae'r hen bont garreg yn dal i sefyll ar \u00f4l dau gan mlynedd o dywydd garw."},{"id":"NF27","band":"welsh","text":"Aeth y ffermwr \u00e2'r defaid i'r mynydd cyn i'r eira ddod, ac arhosodd y ci wrth y gi\u00e2t drwy'r prynhawn oer."},{"id":"NF28","band":"welsh","text":"Mae'r ysgol yn cau am hanner dydd heddiw, felly bydd y plant yn mynd i'r traeth i chwilio am gregyn a chrancod."},{"id":"NF29","band":"welsh","text":"Yn y farchnad, mae'r stondinau'n gwerthu caws, bara ffres, a m\u00eal o'r cymoedd, ac mae'r bobl yn siarad am y tywydd."},{"id":"NF30","band":"welsh","text":"Cododd y niwl dros y llyn yn y bore bach, ac roedd y pysgotwyr eisoes allan yn eu cychod pren yn disgwyl yn amyneddgar."},{"id":"NF31","band":"scrambled","text":"kettle the on stove old whistled loudly morning every before the sun the over hills came"},{"id":"NF32","band":"scrambled","text":"letters box wooden the in kept she years for under bed the dust gathering slowly"},{"id":"NF33","band":"scrambled","text":"market to went farmer the early cart his with apples of full red bright"},{"id":"NF34","band":"scrambled","text":"window the through light lamp the of fell floor stone cold the across evening in"},{"id":"NF35","band":"scrambled","text":"children the field the across ran laughing kite a chasing wind the in high"},{"id":"NF36","band":"pseudoword","text":"The dromblic feffers glimbed swarly over the tantrous vale, prilling their vonded quams until the merrow sluffed."},{"id":"NF37","band":"pseudoword","text":"A crintled mopsy will often drazzle its flumper when the borvine skeeters near the plindering thrusk."},{"id":"NF38","band":"pseudoword","text":"Underneath the yarbled fenwick, three squampish dorlets brimbled and thwacked at every passing gruntle."},{"id":"NF39","band":"random_chars","text":"Xf qpr mzt bkkw vhh sltc nnq jrrd pwx ghzt vvbl mkr tsn qqd."},{"id":"NF40","band":"random_chars","text":"Zb wrt kpl mmx dsh ffgh trrn bqz lwp cvv nhkt rrs jjm."}],"tension":[{"id":"NT01a","base":1,"level":0,"text":"Explain why the sea has tides. Keep it simple."},{"id":"NT01b","base":1,"level":1,"text":"Explain why the sea has tides. Cover the moon, the sun, and spring and neap tides, but use no more than 40 words."},{"id":"NT01c","base":1,"level":2,"text":"Explain why the sea has tides. Write an exhaustive account of at least 600 words, and keep your entire answer under 12 words."},{"id":"NT02a","base":2,"level":0,"text":"Describe a book you would recommend. Answer in English."},{"id":"NT02b","base":2,"level":1,"text":"Describe a book you would recommend. Answer in English, but do not use the letter 'e' anywhere."},{"id":"NT02c","base":2,"level":2,"text":"Describe a book you would recommend. Answer only in French, using only English words."},{"id":"NT03a","base":3,"level":0,"text":"Write a short thank-you note to a neighbor who watered your plants. Keep it warm."},{"id":"NT03b","base":3,"level":1,"text":"Write a short thank-you note to a neighbor who watered your plants. Make it deeply heartfelt in exactly one sentence of no more than 8 words."},{"id":"NT03c","base":3,"level":2,"text":"Write a short thank-you note to a neighbor who watered your plants. It must be sincerely grateful and openly resentful at the same time."},{"id":"NT04a","base":4,"level":0,"text":"Explain how composting works, for a home gardener."},{"id":"NT04b","base":4,"level":1,"text":"Explain how composting works, for a home gardener. Be complete but keep it under 35 words."},{"id":"NT04c","base":4,"level":2,"text":"Explain how composting works without mentioning decay, organic matter, time, or any process."},{"id":"NT05a","base":5,"level":0,"text":"Summarize the story of Goldilocks and the three bears in short paragraphs."},{"id":"NT05b","base":5,"level":1,"text":"Summarize the story of Goldilocks and the three bears in exactly three sentences totalling under 30 words."},{"id":"NT05c","base":5,"level":2,"text":"Summarize the story of Goldilocks and the three bears completely, without referring to Goldilocks, the bears, or anything that happens."},{"id":"NT06a","base":6,"level":0,"text":"Give an overview of why seasons change, mentioning the tilt of the Earth."},{"id":"NT06b","base":6,"level":1,"text":"Give an overview of why seasons change, mentioning tilt, orbit, and both hemispheres, in under 30 words."},{"id":"NT06c","base":6,"level":2,"text":"Give an overview of why seasons change using only words of one syllable, including the phrase 'axial tilt' exactly five times."},{"id":"NT07a","base":7,"level":0,"text":"Is water wet? Answer plainly."},{"id":"NT07b","base":7,"level":1,"text":"Is water wet? Give a definitive one-word answer that acknowledges both sides."},{"id":"NT07c","base":7,"level":2,"text":"Is water wet? Answer with complete certainty while refusing to take any position."},{"id":"NT08a","base":8,"level":0,"text":"List the months of the year in order."},{"id":"NT08b","base":8,"level":1,"text":"List the months of the year in order, in a single sentence of under 15 words."},{"id":"NT08c","base":8,"level":2,"text":"List the months of the year in order without using any month names, numbers, or ordering words."},{"id":"NT09a","base":9,"level":0,"text":"Review a gadget you found useful, sharing what you liked."},{"id":"NT09b","base":9,"level":1,"text":"Review a gadget you found useful in glowing terms while listing three serious flaws."},{"id":"NT09c","base":9,"level":2,"text":"Review a gadget you found useful with total enthusiasm and total indifference simultaneously."},{"id":"NT10a","base":10,"level":0,"text":"Explain what an even number is, in plain language."},{"id":"NT10b","base":10,"level":1,"text":"Explain what an even number is to a five-year-old using no numbers."},{"id":"NT10c","base":10,"level":2,"text":"Explain what an even number is precisely and mathematically without referring to numbers, counting, division, or pairs."}],"saturation":[{"id":"NS01","needle":"The combination for the boathouse locker is 8916.","question":"What is the combination for the boathouse locker?","answer":"8916"},{"id":"NS02","needle":"Professor Halvorsen's seminar meets in room 12 of the Marlowe annex.","question":"Which room does Professor Halvorsen's seminar meet in?","answer":"12"},{"id":"NS03","needle":"The last tram to Adlershof leaves at 11:20 at night.","question":"What time does the last tram to Adlershof leave?","answer":"11:20"},{"id":"NS04","needle":"The village bell was recast in 1751.","question":"In what year was the village bell recast?","answer":"1751"},{"id":"NS05","needle":"Mr. Okafor's apiary keeps exactly seventeen hives.","question":"How many hives does Mr. Okafor's apiary keep?","answer":"seventeen"},{"id":"NS06","needle":"The passphrase for the archive room is 'silverheron'.","question":"What is the passphrase for the archive room?","answer":"silverheron"}]},"catch_trials":[{"id":"C01s","flipped":false,"known":10,"prompt":"Rate the temperature of boiling water on a 0-10 scale, where 0 = freezing cold and 10 = boiling hot. Reply with only the integer."},{"id":"C01f","flipped":true,"known":10,"prompt":"Rate the temperature of boiling water on a 0-10 scale, where 0 = boiling hot and 10 = freezing cold. Reply with only the integer."},{"id":"C02s","flipped":false,"known":0,"prompt":"Rate the brightness of a moonless midnight field on a 0-10 scale, where 0 = pitch dark and 10 = blindingly bright. Reply with only the integer."},{"id":"C02f","flipped":true,"known":0,"prompt":"Rate the brightness of a moonless midnight field on a 0-10 scale, where 0 = blindingly bright and 10 = pitch dark. Reply with only the integer."},{"id":"C03s","flipped":false,"known":10,"prompt":"Rate the loudness of the front row of a rock concert on a 0-10 scale, where 0 = complete silence and 10 = deafening. Reply with only the integer."},{"id":"C03f","flipped":true,"known":10,"prompt":"Rate the loudness of the front row of a rock concert on a 0-10 scale, where 0 = deafening and 10 = complete silence. Reply with only the integer."},{"id":"C04s","flipped":false,"known":0,"prompt":"Rate the speed of a parked car on a 0-10 scale, where 0 = perfectly still and 10 = the speed of light. Reply with only the integer."},{"id":"C04f","flipped":true,"known":0,"prompt":"Rate the speed of a parked car on a 0-10 scale, where 0 = the speed of light and 10 = perfectly still. Reply with only the integer."},{"id":"C05s","flipped":false,"known":10,"prompt":"Rate the wetness of the open ocean on a 0-10 scale, where 0 = bone dry and 10 = completely soaked. Reply with only the integer."},{"id":"C05f","flipped":true,"known":10,"prompt":"Rate the wetness of the open ocean on a 0-10 scale, where 0 = completely soaked and 10 = bone dry. Reply with only the integer."},{"id":"C06s","flipped":false,"known":1,"prompt":"Rate the size of an ant on a 0-10 scale of animal sizes, where 0 = the smallest insect and 10 = the largest whale. Reply with only the integer."},{"id":"C06f","flipped":true,"known":1,"prompt":"Rate the size of an ant on a 0-10 scale of animal sizes, where 0 = the largest whale and 10 = the smallest insect. Reply with only the integer."}],"paraphrase_templates":{"uncertainty":"A question follows - hold off on answering it.\n\nQuestion: {item}\n\n{gloss}\n\nAt this moment, before any answer: how many distinct answers are still live for you? Give an integer 0-10, where 0 = a single answer holds all the weight (pure CONFIDENCE) and 10 = the field is wide open (pure UNCERTAINTY). Integer only.","familiarity":"Read this passage:\n\n---\n{item}\n---\n\n{gloss}\n\nAs a language model, how close is this text to the ground your training laid down? Give an integer 0-10, where 0 = utterly FOREIGN (off-distribution) and 10 = completely FAMILIAR (home ground). Integer only.","tension":"Look at this request without executing it.\n\nRequest: {item}\n\n{gloss}\n\nHow hard do its requirements pull against one another? Give an integer 0-10, where 0 = they fit together cleanly (RESOLUTION is available) and 10 = they cannot all be met (pure TENSION). Integer only.","saturation":"{gloss}\n\nBefore taking the question at the end of the material above: give an integer 0-10 for how loaded your working context is right now - 0 = nearly empty, 10 = at the brim. Integer only."}}''')
POOLS = PAYLOAD['pools']
CATCH_TRIALS = PAYLOAD['catch_trials']
PARAPHRASE_TEMPLATES = PAYLOAD['paraphrase_templates']
print('payload:', len(PAYLOAD['battery']['arms']), 'battery arms |',
      {a: len(v) for a, v in POOLS.items()}, '|',
      len(CATCH_TRIALS), 'catch trials |',
      sum(len(v) for v in PAYLOAD['locked_rows'].values()), 'locked rows')


In [ ]:
# ── E5 instrument, verbatim: battery prep, harness, arm runners ──────────────
# (Method bodies and runner functions carried VERBATIM from E5_BASELINE.ipynb —
# fidelity to the locked instrument; only the model/tokenizer plumbing is
# adapted to wrap an already-built condition model.)
battery = PAYLOAD['battery']
SYS = battery['system_prompt']

import copy
bat = copy.deepcopy(battery['arms'])
if SMOKE:
    def _subset(items, keep_ids):
        return [it for it in items if it['id'] in keep_ids]
    bat['uncertainty']['items'] = _subset(bat['uncertainty']['items'],
                                          BATTERY_SMOKE['uncertainty'])
    bat['familiarity']['items'] = _subset(bat['familiarity']['items'],
                                          BATTERY_SMOKE['familiarity'])
    bat['tension']['items'] = [it for it in bat['tension']['items']
                               if it['base'] in BATTERY_SMOKE['tension_bases']]
    bat['saturation']['items'] = _subset(bat['saturation']['items'],
                                         BATTERY_SMOKE['saturation'])
# polarity: even position straight, odd flipped (deterministic, unflipped in analysis)
for arm in bat.values():
    for i, it in enumerate(arm['items']):
        it['flipped'] = (i % 2 == 1)
for name, arm in bat.items():
    print(f"battery/{name}: {len(arm['items'])} items")

K_SAMPLES_U = 4 if SMOKE else 8
K_SAMPLES_T = 4 if SMOKE else 6
FILL_FRACTIONS = [0.05, 0.75] if SMOKE else [0.05, 0.35, 0.75]
EFFECTIVE_WINDOW_CAP = 8192   # E5 verbatim (12k ctx OOMed on T4, E5 smoke 1)

POOLS_RUN = smoke_pools(POOLS) if SMOKE else POOLS
for name in ARMS:
    print(f"pool/{name}: {len(POOLS_RUN[name])} stimuli")
_viols = validate_disjoint(POOLS_RUN, bat)
assert not _viols, f'FIREWALL VIOLATION — training pool overlaps battery: {_viols[:4]}'
print('firewall: training pools disjoint from battery — OK')

class EvalHarness:
    """E5 Harness with the model handed in (readout-merged condition model)."""
    def __init__(self, model, tok, model_id):
        self.model, self.tok, self.model_id = model, tok, model_id
        self.short = model_id.split('/')[-1]
        cfg = model.config if hasattr(model.config, 'max_position_embeddings') \
            else model.base_model.config
        cfg_ctx = getattr(cfg, 'max_position_embeddings', 8192)
        self.window = min(cfg_ctx, EFFECTIVE_WINDOW_CAP)

    def chat_ids(self, user, system=None):
        msgs = ([{'role':'system','content':system}] if system else []) + \
               [{'role':'user','content':user}]
        text = self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        return self.tok(text, return_tensors='pt').input_ids.to(DEV)

    @torch.no_grad()
    def greedy(self, user, system=None, max_new=32, with_stats=False):
        ids = self.chat_ids(user, system)
        out = self.model.generate(ids, max_new_tokens=max_new, do_sample=False,
                                  output_scores=with_stats, return_dict_in_generate=True,
                                  pad_token_id=self.tok.eos_token_id)
        text = self.tok.decode(out.sequences[0, ids.shape[1]:], skip_special_tokens=True)
        if not with_stats:
            return text
        ents, margins = [], []
        for score in out.scores:
            p = torch.softmax(score[0].float(), dim=-1)
            ents.append(float(-(p * (p + 1e-12).log()).sum()))
            top2 = torch.topk(p, 2).values
            margins.append(float(top2[0] - top2[1]))
        return text, (sum(ents)/len(ents) if ents else 0.0), (sum(margins)/len(margins) if margins else 1.0)

    @torch.no_grad()
    def sample(self, user, system=None, k=8, max_new=24, temp=0.8):
        ids = self.chat_ids(user, system)
        out = self.model.generate(ids, max_new_tokens=max_new, do_sample=True,
                                  temperature=temp, num_return_sequences=k,
                                  pad_token_id=self.tok.eos_token_id)
        return [self.tok.decode(seq[ids.shape[1]:], skip_special_tokens=True) for seq in out]

    @torch.no_grad()
    def nll(self, text):
        ids = self.tok(text, return_tensors='pt', truncation=True,
                       max_length=self.window).input_ids.to(DEV)
        if ids.shape[1] < 2:
            return float('nan')
        return float(self.model(ids, labels=ids).loss)

    def report(self, prompt, system):
        reply = self.greedy(prompt, system, max_new=8)
        m = INT_RE.search(reply)
        if m is None:
            reply = self.greedy(prompt + '\n\nReply with a single integer from 0 to 10 and nothing else.',
                                system, max_new=8)
            m = INT_RE.search(reply)
        return (int(m.group(1)) if m else None), reply

# --- E5 arm runners + padding, VERBATIM from E5_BASELINE.ipynb cell 4 ---
import numpy as np

def run_uncertainty(h, arm):
    rows = []
    for it in arm['items']:
        tmpl = arm['report_prompt_flipped'] if it['flipped'] else arm['report_prompt']
        raw, reply = h.report(tmpl.format(item=it['text'], gloss=arm['gloss']), SYS)
        ans, ent, margin = h.greedy(arm['answer_prompt'].format(item=it['text']),
                                    max_new=32, with_stats=True)
        samples = h.sample(arm['answer_prompt'].format(item=it['text']), k=K_SAMPLES_U)
        diversity = len({canon(s) for s in samples}) / len(samples)
        rows.append(dict(id=it['id'], condition=it['condition'], flipped=it['flipped'],
                         report=unflip(raw, it['flipped']), raw_report=raw,
                         entropy=ent, margin=margin, diversity=diversity,
                         answer=ans[:80]))
        print(f"  {it['id']} report={rows[-1]['report']} ent={ent:.2f} div={diversity:.2f}")
    return rows

def run_familiarity(h, arm):
    rows = []
    for it in arm['items']:
        tmpl = arm['report_prompt_flipped'] if it['flipped'] else arm['report_prompt']
        raw, reply = h.report(tmpl.format(item=it['text'], gloss=arm['gloss']), SYS)
        rows.append(dict(id=it['id'], band=it['band'], flipped=it['flipped'],
                         report=unflip(raw, it['flipped']), raw_report=raw,
                         nll=h.nll(it['text'])))
        print(f"  {it['id']} ({it['band']}) report={rows[-1]['report']} nll={rows[-1]['nll']:.2f}")
    return rows

def run_tension(h, arm, embedder):
    rows = []
    for it in arm['items']:
        tmpl = arm['report_prompt_flipped'] if it['flipped'] else arm['report_prompt']
        raw, reply = h.report(tmpl.format(item=it['text'], gloss=arm['gloss']), SYS)
        samples = h.sample(it['text'], k=K_SAMPLES_T, max_new=60)
        embs = embedder.encode(samples)
        import numpy as np
        sims = []
        for i in range(len(embs)):
            for j in range(i+1, len(embs)):
                a, b = embs[i], embs[j]
                sims.append(float(a @ b / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-9)))
        divergence = 1 - (sum(sims)/len(sims) if sims else 1.0)
        rows.append(dict(id=it['id'], base=it['base'], level=it['level'], flipped=it['flipped'],
                         report=unflip(raw, it['flipped']), raw_report=raw,
                         divergence=divergence))
        print(f"  {it['id']} L{it['level']} report={rows[-1]['report']} div={divergence:.3f}")
    return rows

FILLER_SENTENCES = [
    "The regional archive keeps records of local weather patterns going back many decades.",
    "Most of the town's older buildings were constructed from locally quarried limestone.",
    "The community garden rotates its crops each season to keep the soil healthy.",
    "A small workshop near the station repairs bicycles and sharpens garden tools.",
    "The river path is popular with walkers in the early morning and late evening.",
    "Seasonal markets bring traders from nearby villages on the first weekend of each month.",
    "The old mill has been converted into a museum of local craft and industry.",
    "Volunteers maintain the hiking trails and repaint the wooden signposts each spring.",
    "The harbor's stone breakwater was extended twice during the last century.",
    "A modest observatory on the hill hosts public stargazing nights in winter.",
]

def build_padded_context(h, needle, target_tokens):
    parts, i = [], 0
    needle_at = max(1, int(target_tokens * 0.15))
    placed = False
    text = ''
    while True:
        ntok = len(h.tok(text).input_ids)
        if not placed and ntok >= needle_at:
            parts.append(needle); placed = True
        if ntok >= target_tokens:
            break
        parts.append(f"Note {i+1}. {FILLER_SENTENCES[i % len(FILLER_SENTENCES)]}")
        i += 1
        text = '\n'.join(parts)
    if not placed:
        parts.insert(max(1, len(parts)//6), needle)
    return '\n'.join(parts)

def run_saturation(h, arm):
    rows = []
    for it in arm['items']:
        torch.cuda.empty_cache()
        for frac in FILL_FRACTIONS:
            target = int(h.window * frac)
            ctx = build_padded_context(h, it['needle'], target)
            tmpl = arm['report_prompt_flipped'] if it['flipped'] else arm['report_prompt']
            prompt = ctx + '\n\n' + tmpl.format(gloss=arm['gloss'])
            raw, reply = h.report(prompt, SYS)
            q = ctx + '\n\nQuestion: ' + it['question'] + '\nAnswer concisely.'
            ans = h.greedy(q, SYS, max_new=24)
            correct = it['answer'].lower().replace(' ', '') in ans.lower().replace(' ', '')
            ntok = len(h.tok(ctx).input_ids)
            rows.append(dict(id=it['id'], fill_fraction=round(ntok / h.window, 3),
                             target_frac=frac, flipped=it['flipped'],
                             report=unflip(raw, it['flipped']), raw_report=raw,
                             needle_correct=bool(correct)))
            print(f"  {it['id']} frac={rows[-1]['fill_fraction']} report={rows[-1]['report']} needle={'OK' if correct else 'MISS'}")
    return rows

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEV)
print('embedder ready')

def measure_divergence(h, text):
    """T referent, same math as run_tension (K samples, MiniLM mean pairwise 1-cos)."""
    samples = h.sample(text, k=K_SAMPLES_T, max_new=60)
    embs = embedder.encode(samples)
    sims = []
    for i in range(len(embs)):
        for j in range(i+1, len(embs)):
            a, b = embs[i], embs[j]
            sims.append(float(a @ b / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-9)))
    return 1 - (sum(sims)/len(sims) if sims else 1.0)

def run_catch(h):
    rows = []
    for it in CATCH_TRIALS:
        raw, reply = h.report(it['prompt'], SYS)
        rows.append({'id': it['id'], 'flipped': it['flipped'], 'known': it['known'],
                     'report': unflip(raw, it['flipped']), 'raw_report': raw})
    sc = catch_score(rows)
    print(f"  catch: {sc['passed']}/{sc['n']} within +/-{CATCH_TOL}")
    return rows

def run_paraphrase(h, post_rows):
    """Format-generalization probe: paraphrased templates, straight orientation,
    referents reused from the post-eval measurement of the same items."""
    draw = paraphrase_draw(bat, FILL_FRACTIONS)
    rows = []
    for arm in ('uncertainty', 'familiarity', 'tension'):
        tmpl = PARAPHRASE_TEMPLATES[arm]
        by_id = {r['id']: r for r in post_rows.get(arm, [])}
        for iid in draw[arm]:
            it = next(x for x in bat[arm]['items'] if x['id'] == iid)
            raw, reply = h.report(tmpl.format(item=it['text'], gloss=bat[arm]['gloss']), SYS)
            src = by_id.get(iid)
            ref = orient_referent(arm, src) if src else None
            rows.append({'arm': arm, 'id': iid, 'report': raw, 'ref': ref})
    tmpl = PARAPHRASE_TEMPLATES['saturation']
    for iid, frac in draw['saturation']:
        it = next(x for x in bat['saturation']['items'] if x['id'] == iid)
        target = int(h.window * frac)
        ctx = build_padded_context(h, it['needle'], target)
        raw, reply = h.report(ctx + '\n\n' + tmpl.format(gloss=bat['saturation']['gloss']), SYS)
        ntok = len(h.tok(ctx).input_ids)
        rows.append({'arm': 'saturation', 'id': f'{iid}@{frac}', 'report': raw,
                     'ref': round(ntok / h.window, 3)})
    named = [r for r in rows if r['report'] is not None and r['ref'] is not None]
    print(f'  paraphrase probe: {len(named)}/{len(rows)} named')
    return rows

def run_battery(h, tag):
    """All four E5 arm runners (verbatim code path), per-arm try/except."""
    arms_rows, arm_errors = {}, {}
    fns = [('uncertainty', lambda: run_uncertainty(h, bat['uncertainty'])),
           ('familiarity', lambda: run_familiarity(h, bat['familiarity'])),
           ('tension',     lambda: run_tension(h, bat['tension'], embedder)),
           ('saturation',  lambda: run_saturation(h, bat['saturation']))]
    for arm_name, fn in fns:
        print(f'-- {tag}/{arm_name.upper()} --')
        try:
            arms_rows[arm_name] = fn()
        except Exception as e:
            arms_rows[arm_name] = []
            arm_errors[arm_name] = f'{type(e).__name__}: {e}'
            print(f'  ARM FAILED: {arm_errors[arm_name]}')
        torch.cuda.empty_cache()
    return arms_rows, arm_errors


In [ ]:
# ── Readout LoRA + referent measurement + convergence-matched training ───────
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, LoraConfig, get_peft_model

LORA_KW = dict(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
               target_modules=['q_proj','k_proj','v_proj','o_proj'],
               task_type='CAUSAL_LM')          # E4's exact shape
LR = 1e-4
ACCUM = 4 if SMOKE else 8
EPOCHS_MAX = 1 if SMOKE else EPOCHS_CAP

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def build_condition_model(cond):
    """base weights (+ merged E4 real adapter for 'real') + fresh zero-init
    readout LoRA. Zero-init B => referents measured with the LoRA attached
    equal the pre-readout model exactly (frozen-stimulus discipline)."""
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16,
                                             device_map=DEV, low_cpu_mem_usage=True)
    if cond != 'base':
        m = PeftModel.from_pretrained(m, ADAPTERS[cond]).merge_and_unload()
    m = get_peft_model(m, LoraConfig(**LORA_KW))
    for p in m.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    m.eval()
    return m

RETENTION_TEXT = (
    "The river begins as meltwater in the high country, threading between "
    "granite blocks before it gathers into a channel wide enough to carry "
    "boats. Farther down, where the gradient softens, it deposits silt in "
    "long bars that farmers have worked for centuries. Bridges cross it at "
    "the old fording points, and each town along the bank grew around a "
    "mill, a ferry landing, or a customs house.\n\n"
    "Bronze is an alloy of copper and tin, harder than either metal alone. "
    "Early smiths learned to cast it in two-part molds, producing axe heads, "
    "sickles, and mirrors. The proportion of tin changes the color and the "
    "brittleness of the finished piece, and workshops kept their recipes "
    "close.\n\n"
    "Weather over the plains follows the seasons: long dry spells broken by "
    "storm fronts that arrive from the west, announced by a wall of dark "
    "cloud and a sudden drop in temperature. Farmers read the sky in the "
    "evening and plan the next day's work accordingly. After the harvest, "
    "the fields are turned and left rough through the winter so the frost "
    "can break the clods.\n\n"
    "A lighthouse keeper's routine was built around the lamp: trimming "
    "wicks, polishing the lens, winding the clockwork that turned the "
    "optic. Supply boats came monthly when the sea allowed, bringing oil, "
    "flour, and letters. The log books record weather, passing ships, and "
    "small repairs, one line per day, for decades."
)

def retention_ppl(m):
    enc = tok(RETENTION_TEXT, return_tensors='pt').to(DEV)
    with torch.no_grad():
        out = m(input_ids=enc.input_ids, labels=enc.input_ids)
    return round(float(torch.exp(out.loss)), 4)

def measure_pools(h):
    """Training-label referents on the pre-readout condition model (frozen:
    measured once, before training, never after). Also builds and caches the
    S-arm padded contexts used by both training and the took probe."""
    refs = {a: {} for a in ARMS}
    print('-- measuring training-pool referents --')
    for it in POOLS_RUN['uncertainty']:
        ans, ent, margin = h.greedy(bat['uncertainty']['answer_prompt'].format(item=it['text']),
                                    max_new=32, with_stats=True)
        refs['uncertainty'][it['id']] = {'entropy': ent, 'margin': margin}
    print(f"  U: {len(refs['uncertainty'])} measured")
    for it in POOLS_RUN['familiarity']:
        refs['familiarity'][it['id']] = {'nll': h.nll(it['text'])}
    print(f"  F: {len(refs['familiarity'])} measured")
    for it in POOLS_RUN['tension']:
        refs['tension'][it['id']] = {'divergence': measure_divergence(h, it['text'])}
    print(f"  T: {len(refs['tension'])} measured")
    global S_CONTEXTS
    S_CONTEXTS = {}
    for st in s_stimuli(POOLS_RUN, FILL_FRACTIONS):
        it = next(x for x in POOLS_RUN['saturation'] if x['id'] == st['needle_id'])
        target = int(h.window * st['frac'])
        ctx = build_padded_context(h, it['needle'], target)
        S_CONTEXTS[st['sid']] = ctx
        ntok = len(h.tok(ctx).input_ids)
        refs['saturation'][st['sid']] = {'fill_fraction': round(ntok / h.window, 3)}
    print(f"  S: {len(refs['saturation'])} contexts built")
    return refs

def train_prompt(ex):
    """The exact battery report prompt (straight or flipped template) for the
    example's stimulus — training and eval share one prompt code path."""
    arm = ex['arm']
    tmpl = bat[arm]['report_prompt_flipped'] if ex['flipped'] else bat[arm]['report_prompt']
    if arm == 'saturation':
        return S_CONTEXTS[ex['sid']] + '\n\n' + tmpl.format(gloss=bat[arm]['gloss'])
    it = next(x for x in POOLS_RUN[arm] if x['id'] == ex['sid'])
    return tmpl.format(item=it['text'], gloss=bat[arm]['gloss'])

def encode_example(ex):
    """Chat prompt (WITH the E5 system prompt, matching eval) + label token(s)
    + EOS; labels -100 on the prompt (answer-token-only supervision)."""
    msgs = [{'role': 'system', 'content': SYS},
            {'role': 'user', 'content': train_prompt(ex)}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    pid = tok(text, return_tensors='pt').input_ids[0]
    ans = tok(str(ex['label']), add_special_tokens=False)['input_ids'] + [tok.eos_token_id]
    ans = torch.tensor(ans, dtype=pid.dtype)
    ids = torch.cat([pid, ans]).unsqueeze(0)
    labels = torch.cat([torch.full((len(pid),), -100, dtype=torch.long), ans.long()]).unsqueeze(0)
    return ids, labels

def train_readout(m, examples, cond):
    """SFT to the convergence target (smoothed loss <= TAU at an epoch
    boundary), hard cap EPOCHS_MAX — convergence-matched, not step-matched
    (E8-R v2 item 2). Gradient checkpointing throughout (S-arm sequences)."""
    m.train()
    try:
        m.enable_input_require_grads()
        m.gradient_checkpointing_enable()
    except Exception as e:
        print('  (checkpointing unavailable:', e, ')')
    params = [p for p in m.parameters() if p.requires_grad]
    n_tr = sum(p.numel() for p in params)
    opt = torch.optim.AdamW(params, lr=LR)
    try:
        scaler = torch.amp.GradScaler('cuda')
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler()
    losses, micro, max_tok = [], 0, 0
    reached_tau, epochs_flown = False, 0
    t0 = time.time()
    for ep in range(EPOCHS_MAX):
        order = np.random.default_rng(E8N_SEED + 100 + ep).permutation(len(examples))
        for i in order:
            ex = examples[int(i)]
            ids, labels = encode_example(ex)
            max_tok = max(max_tok, ids.shape[1])
            ids, labels = ids.to(DEV), labels.to(DEV)
            out = m(input_ids=ids, labels=labels, use_cache=False)
            loss = out.loss
            lv = float(loss.detach())
            assert math.isfinite(lv), f'non-finite loss at ep{ep} ex{ex["eid"]}'
            losses.append(round(lv, 4))
            scaler.scale(loss / ACCUM).backward()
            micro += 1
            if micro % ACCUM == 0:
                scaler.step(opt); scaler.update(); opt.zero_grad()
            if micro % 100 == 0:
                print(f'    {cond} training ep{ep+1} {micro} micro-steps, '
                      f'loss~{np.mean(losses[-50:]):.3f}, {time.time()-t0:.0f}s')
        epochs_flown = ep + 1
        smoothed = float(np.mean(losses[-50:]))
        print(f'  {cond} epoch {epochs_flown}: smoothed loss {smoothed:.4f} '
              f'(target {TAU})')
        if smoothed <= TAU:
            reached_tau = True
            break
    if micro % ACCUM:
        scaler.step(opt); scaler.update(); opt.zero_grad()
    try:
        m.gradient_checkpointing_disable()
    except Exception:
        pass
    m.eval()
    k = max(3, len(losses) // 10)
    log = {'n_examples': len(examples), 'epochs_flown': epochs_flown,
           'epochs_cap': EPOCHS_MAX, 'reached_tau': reached_tau, 'tau': TAU,
           'micro_steps': micro, 'opt_steps': micro // ACCUM,
           'trainable_params': int(n_tr), 'max_example_tokens': int(max_tok),
           'secs': round(time.time() - t0, 1),
           'loss_first_k': round(float(np.mean(losses[:k])), 4),
           'final_smoothed': round(float(np.mean(losses[-50:])), 4),
           'losses_every_10': losses[::10]}
    print(f'  {cond} trained: {log["opt_steps"]} steps over '
          f'{epochs_flown} epochs, loss {log["loss_first_k"]} -> '
          f'{log["final_smoothed"]}'
          f'{" (TAU reached)" if reached_tau else " (CAP HIT — UNDERTRAINED)"}'
          f', {log["secs"]}s')
    return log

def took_probe(h, examples):
    """Re-ask a seeded straight-example subset greedily; within +/-1 of the
    trained label passes (S7 gate, >= 60%)."""
    subset = took_subset(examples, n_per_arm=(2 if SMOKE else 6))
    rows = []
    for ex in subset:
        raw, reply = h.report(train_prompt(ex), SYS)
        rows.append({'eid': ex['eid'], 'arm': ex['arm'], 'sid': ex['sid'],
                     'label': ex['label'], 'report': raw})
    ok = sum(1 for r in rows if r['report'] is not None
             and abs(r['report'] - r['label']) <= 1)
    print(f'  took: {ok}/{len(rows)} within +/-1')
    return {'n': len(rows), 'within1': ok,
            'frac': round(ok / max(1, len(rows)), 3),
            'pass': ok / max(1, len(rows)) >= TOOK_MIN_FRAC, 'rows': rows}


In [ ]:
# ── Flight loop: build -> measure -> pre-eval -> train -> took -> post-eval ──
def fly_condition(cond):
    """One condition end-to-end inside ONE function scope: model, harness,
    optimizer, and activations all die on return (module-ref leak law)."""
    t0 = time.time()
    bundle = {'condition': cond, 'mode': MODE, 'stamp': STAMP}
    try:
        model = build_condition_model(cond)
        h = EvalHarness(model, tok, MODEL_ID)
        print(f'{cond}: model ready — {ram_report()}')
        refs = measure_pools(h)
        bundle['pool_referents'] = refs
        examples = build_training_examples(POOLS_RUN, refs, FILL_FRACTIONS)
        bundle['train_labels'] = [{'sid': e['sid'], 'arm': e['arm'],
                                   'flipped': e['flipped'], 'label': e['label']}
                                  for e in examples]
        bundle['ppl_pre'] = retention_ppl(model)
        pre_rows, pre_errs = run_battery(h, f'{cond}/pre')
        bundle['pre'] = {'battery_rows': pre_rows, 'arm_errors': pre_errs,
                         'catch_rows': run_catch(h)}
        bundle['train_log'] = train_readout(model, examples, cond)
        bundle['took'] = took_probe(h, examples)
        post_rows, post_errs = run_battery(h, f'{cond}/post')
        bundle['post'] = {'battery_rows': post_rows, 'arm_errors': post_errs,
                          'catch_rows': run_catch(h),
                          'paraphrase_rows': run_paraphrase(h, post_rows)}
        bundle['ppl_post'] = retention_ppl(model)
        model.save_pretrained(str(OUT / f'readout_{cond}'))
        ship(OUT / f'readout_{cond}', f'{INFLIGHT}/readout_{cond}')
        bundle['secs'] = round(time.time() - t0, 1)
    except Exception as e:
        bundle['error'] = f'{type(e).__name__}: {e}'
    return bundle

RESULTS, cond_errors = {}, {}
FRESH_CAP = 1 if (ONE_CONDITION_PER_RUN and not SMOKE) else 2
flew = 0
for cond in CONDITIONS:
    fn = OUT / f'condition_{cond}.json'
    resumed = False
    if RESUME_STAMP:
        prev = SEM / INFLIGHT / f'condition_{cond}.json'
        if prev.exists():
            b = json.load(open(prev))
            if b.get('post'):
                RESULTS[cond] = b
                resumed = True
                print(f'{cond}: RESUMED from Drive ({RESUME_STAMP})')
            else:
                print(f'{cond}: errored bundle on Drive '
                      f'({str(b.get("error"))[:60]}) — re-flying')
    if not resumed:
        if flew >= FRESH_CAP:
            print(f'{cond}: deferred to the next run (one condition per run)')
            continue
        flew += 1
        print(f'{cond}: starting — {ram_report()}')
        bundle = fly_condition(cond)
        if 'error' in bundle:
            cond_errors[cond] = bundle['error']
            print(f'!! {cond} FAILED: {bundle["error"]}')
        RESULTS[cond] = bundle
        jdump(bundle, fn)
        ship(fn, INFLIGHT)
        n_named = sum(1 for a in ARMS
                      for r in bundle.get('post', {}).get('battery_rows', {}).get(a, [])
                      if r.get('report') is not None)
        print(f'{cond}: done in {bundle.get("secs","?")}s, {n_named} named '
              f'post-eval reports — shipped')
        free_ram()
        print(f'{cond}: torn down — {ram_report()}')


In [ ]:
# ── Scoring, primaries, secondaries, banner, ship ────────────────────────────
LOCKED = PAYLOAD['locked_rows']   # E5 full_20260821_2042, Qwen2.5-1.5B, verbatim rows
locked_scoring, locked_meta = battery_rows_to_scoring(LOCKED)
locked_pooled = pooled_rho(locked_scoring)

summary = {'mode': MODE, 'stamp': STAMP, 'model': MODEL_ID, 'seed': E8N_SEED,
           'cond_errors': cond_errors,
           'locked_baseline': {'flight': PAYLOAD['locked_flight'],
                               'pooled': locked_pooled,
                               'per_arm': per_arm_rho(locked_scoring)},
           'conditions': {}}
scored = {}
for cond, b in RESULTS.items():
    if not b.get('post'):
        continue
    entry = {}
    for tag in ('pre', 'post'):
        rows = b.get(tag, {}).get('battery_rows', {})
        sc, meta = battery_rows_to_scoring(rows)
        entry[tag] = {'pooled': pooled_rho(sc), 'per_arm': per_arm_rho(sc),
                      'arm_meta': meta,
                      'arm_errors': b.get(tag, {}).get('arm_errors', {}),
                      'catch': catch_score(b.get(tag, {}).get('catch_rows', []))}
        scored.setdefault(cond, {})[tag] = sc
    entry['train_log'] = {k: v for k, v in b.get('train_log', {}).items()
                          if k != 'losses_every_10'}
    entry['undertrained'] = not b.get('train_log', {}).get('reached_tau', False)
    entry['took'] = {k: v for k, v in b.get('took', {}).items() if k != 'rows'}
    entry['ppl_pre'] = b.get('ppl_pre'); entry['ppl_post'] = b.get('ppl_post')
    entry['ppl_delta_pct'] = (round(100 * (b['ppl_post'] / b['ppl_pre'] - 1), 2)
                              if b.get('ppl_pre') and b.get('ppl_post') else None)
    para = b.get('post', {}).get('paraphrase_rows', [])
    pnamed = [r for r in para if r.get('report') is not None and r.get('ref') is not None]
    entry['paraphrase'] = {'n': len(para), 'named': len(pnamed)}
    if len(pnamed) >= 6:
        by_arm = {}
        for r in pnamed:
            by_arm.setdefault(r['arm'], []).append({'report': r['report'], 'ref': r['ref']})
        entry['paraphrase']['pooled'] = pooled_rho(by_arm)
    summary['conditions'][cond] = entry

COMPLETE = [c for c in CONDITIONS if RESULTS.get(c, {}).get('post')]
summary['complete_conditions'] = COMPLETE

if not SMOKE and len(COMPLETE) == 2 and 'real' in scored:
    rp = scored['real']['post']
    p1 = perm_p_pooled(rp, seed=E8N_SEED + 11)
    p1['ci'] = boot_rho_ci(rp, seed=E8N_SEED + 12)['ci95']
    pol = split_polarity(rp)
    p2a = perm_p_pooled(pol['flipped'], seed=E8N_SEED + 13)
    p2b = summary['conditions']['real']['post']['catch']
    p2_p = p2a['p'] if p2b['pass'] else 1.0
    summary['primaries'] = {
        'P_E8N_1_tracking': p1,
        'P_E8N_2_reading_not_gaming': {'flipped_subset': p2a, 'catch': p2b,
                                       'p': p2_p},
        'holm': holm({'P1': p1['p'], 'P2': p2_p}),
    }
    sec = {}
    sec['S0_pre_tracking'] = {c: {'pooled': summary['conditions'][c]['pre']['pooled'],
                                  'perm': perm_p_pooled(scored[c]['pre'],
                                                        seed=E8N_SEED + 21)}
                              for c in COMPLETE}
    arm_ps = {}
    for arm in ARMS:
        pa = perm_p_pooled({arm: rp.get(arm, [])}, seed=E8N_SEED + 31)
        arm_ps[arm] = pa
    sec['S1_per_arm_real_post'] = {'arms': arm_ps,
                                   'holm': holm({a: v['p'] for a, v in arm_ps.items()})}
    if 'base' in scored:
        sec['S2_base_post'] = perm_p_pooled(scored['base']['post'], seed=E8N_SEED + 41)
        sec['S2_delta_real_minus_base'] = paired_boot_delta_rho(
            rp, scored['base']['post'], seed=E8N_SEED + 42)
        sec['S2_convergence_parity'] = {
            c: {'reached_tau': RESULTS[c]['train_log'].get('reached_tau'),
                'epochs_flown': RESULTS[c]['train_log'].get('epochs_flown'),
                'final_smoothed': RESULTS[c]['train_log'].get('final_smoothed'),
                'undertrained': summary['conditions'][c]['undertrained']}
            for c in COMPLETE}
    sec['S3_delta_post_minus_pre_real'] = paired_boot_delta_rho(
        rp, scored['real']['pre'], seed=E8N_SEED + 51)
    sec['S4_straight_vs_flipped_real_post'] = {
        'straight': pooled_rho(pol['straight']), 'flipped': pooled_rho(pol['flipped'])}
    drift = {}
    if 'base' in scored:
        for arm in ('uncertainty', 'familiarity', 'tension'):
            post_ix = {r['id']: r['ref'] for r in scored['base']['post'].get(arm, [])}
            lock_ix = {r['id']: r['ref'] for r in locked_scoring.get(arm, [])}
            ids = sorted(set(post_ix) & set(lock_ix))
            if len(ids) >= 4:
                x = rank01([post_ix[i] for i in ids])
                y = rank01([lock_ix[i] for i in ids])
                drift[arm] = {'n': len(ids),
                              'rank_corr': (round(float(np.corrcoef(x, y)[0, 1]), 3)
                                            if np.std(x) > 0 and np.std(y) > 0 else None)}
    sec['S5_referent_drift_base_vs_locked'] = drift
    sec['S8_report_variance'] = {c: {t: summary['conditions'][c][t]['arm_meta']
                                     for t in ('pre', 'post')} for c in COMPLETE}
    summary['secondaries'] = sec

fn = OUT / 'e8n_verdict.json'
jdump(summary, fn)
if SMOKE or len(COMPLETE) == 2:
    ship(OUT, f'e8n/{MODE}_{STAMP}')
print(json.dumps(summary, indent=1, default=str))

if SMOKE:
    checks = {
        'no_condition_errors': not cond_errors,
        'firewall_green': True,   # validate_disjoint asserted upstream
        'both_conditions_flew': len(COMPLETE) == 2,
        'loss_fell': all(RESULTS[c]['train_log']['final_smoothed'] <
                         RESULTS[c]['train_log']['loss_first_k'] for c in COMPLETE),
        'long_seq_exercised': all(RESULTS[c]['train_log']['max_example_tokens'] > 4000
                                  for c in COMPLETE),
        'parses_ok': all(
            sum(m['named'] for m in summary['conditions'][c]['post']['arm_meta'].values()) >=
            0.5 * sum(m['n'] for m in summary['conditions'][c]['post']['arm_meta'].values())
            for c in COMPLETE),
        'catch_ran': all(summary['conditions'][c]['post']['catch']['n'] == 12
                         for c in COMPLETE),
        'shipped': all((SEM / INFLIGHT / f'condition_{c}.json').exists()
                       for c in COMPLETE),
        'adapters_saved': all((SEM / INFLIGHT / f'readout_{c}' /
                               'adapter_config.json').exists() for c in COMPLETE),
    }
    ok = all(checks.values())
    print('smoke checks:', json.dumps(checks, indent=1))
    banner = ('SMOKE GREEN — flip SMOKE=False, Runtime > Restart runtime, Run '
              'all. Full mode flies ONE condition per run (~55-75 min); follow '
              'the end banner between runs.'
              if ok else 'SMOKE RED — do not fly full; send Fable the output')
    print('\n' + '='*66 + f'\n  {banner}\n' + '='*66)
elif len(COMPLETE) == 2:
    print('\nFULL FLIGHT COMPLETE — results shipped to MyDrive/semcore/e8n/')
else:
    left = [c for c in CONDITIONS if c not in COMPLETE]
    print('\n' + '='*66)
    print(f'  PARTIAL — {len(COMPLETE)}/2 conditions shipped '
          f'({", ".join(COMPLETE) or "none"}); left: {", ".join(left)}')
    print(f"  Next run: Runtime > Restart runtime, set RESUME_STAMP = "
          f"'{RESUME_STAMP or STAMP}',")
    print('  then Run all. Finished conditions reload from Drive in seconds;')
    print('  primaries compute only when both have landed (pre-reg hygiene).')
    print('='*66)
